## Import Libraries and load dataset 

In [1]:
# Import Libraries
import os
import glob
import numpy as np
import torch
import random
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, random_split
import torch.nn.functional as F
import foolbox as fb
from foolbox.attacks import FGSM, PGD, LinfBasicIterativeAttack, BoundaryAttack, L2DeepFoolAttack, L2CarliniWagnerAttack
import numpy as np
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")
import zipfile
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.preprocessing import QuantileTransformer
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from torchsummary import summary
from imblearn.over_sampling import RandomOverSampler
from sklearn.compose import ColumnTransformer
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score, confusion_matrix, ConfusionMatrixDisplay, classification_report
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder, LabelEncoder
import zipfile
import random

In [2]:
df=pd.read_csv("merged_IDS2018.csv")

In [3]:
df.shape

(9625148, 79)

## Clean and Preprocess Dataset 

In [4]:
# Create Class column
df['Label'] = df['Label'].str.strip()
df['Class'] = df['Label'].apply(lambda x: 'Benign' if x == 'Benign' else 'Attack')

In [5]:
# Identify columns where every single row has the same value
constant_cols = [col for col in df.columns if df[col].nunique() <= 1]

# Drop them safely
df.drop(columns=constant_cols, inplace=True)

In [6]:
# columns to drop
columns_to_drop = ['Dst Port', 
                   'Flow Byts/s', 'Flow Pkts/s', 'Fwd Pkts/s', 'Bwd Pkts/s',
                   'Pkt Size Avg', 'Fwd Seg Size Avg', 'Bwd Seg Size Avg']
df.drop(columns=columns_to_drop, inplace=True)

# remove the outliers (negative values) from selected columns
ngtv_cols = ['Flow Duration','Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min',
             'Fwd IAT Tot', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min',
             'Bwd IAT Tot', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min']
for col in ngtv_cols:
    df = df[df[col] >= 0]

# Cap the upper limit of selected columns to their 95th percentile
for i in ['Flow Duration', 
          'Tot Fwd Pkts', 'Tot Bwd Pkts', 
          'TotLen Fwd Pkts', 'TotLen Bwd Pkts', 
          'Fwd Pkt Len Max', 'Fwd Pkt Len Mean', 'Fwd Pkt Len Std',
          'Bwd Pkt Len Max', 'Bwd Pkt Len Mean', 'Bwd Pkt Len Std',
          'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min',
          'Fwd IAT Tot', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min',
          'Bwd IAT Tot', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min',
          'Fwd Header Len', 'Bwd Header Len',
          'Pkt Len Max', 'Pkt Len Mean', 'Pkt Len Std', 'Pkt Len Var',
          'Down/Up Ratio',
          'Subflow Fwd Pkts', 'Subflow Fwd Byts', 'Subflow Bwd Pkts', 'Subflow Bwd Byts',
          'Fwd Act Data Pkts',
          'Active Mean', 'Active Std', 'Active Max', 'Active Min',
           'Idle Mean', 'Idle Std', 'Idle Max', 'Idle Min'
          ]:
    upper_limit = df[i].quantile(0.95)
    df[i] = df[i].clip(upper=upper_limit)

In [7]:
df.shape

(9625138, 64)

In [8]:
# Split features and targets
X = df.drop(columns=['Label', 'Class'])
y_binary = df['Class']
# y_multi = df['Label']

In [9]:
def prepare_data(X, y, test_size=0.2, random_state=42):
    # Split the data
    X_train, X_test, y_train, y_test = train_test_split(
        X, y_binary, test_size=test_size,random_state=random_state,stratify=y_binary
        )
    
    # Split the remaining 20% into Test (50% total) and Validation (50% total)
    X_test, X_val, y_test, y_val = train_test_split(
        X_test, y_test,
        test_size=0.50,
        random_state=random_state,
        stratify=y_test
        )

    ros = RandomOverSampler(
        sampling_strategy='auto',  
        random_state=random_state
        )
    
    X_train, y_train = ros.fit_resample(X_train, y_train)
    
    # Define column groups
    categorical_cols = [
        'Protocol',
        'Fwd PSH Flags','Fwd URG Flags',
        'FIN Flag Cnt','SYN Flag Cnt','RST Flag Cnt','ACK Flag Cnt',
        'URG Flag Cnt','PSH Flag Cnt','ECE Flag Cnt','CWE Flag Count'
        ]
    
    numerical_cols = [c for c in X.columns if c not in categorical_cols]

    # Build the transformer
    preprocessor = ColumnTransformer(
        transformers=[
            ('num', MinMaxScaler(), numerical_cols),
            ('cat', OneHotEncoder(sparse_output=False, handle_unknown='ignore'), categorical_cols)
      ]
    )

    # Apply the transformations and cast to float32 
    X_train = preprocessor.fit_transform(X_train)
    X_val = preprocessor.transform(X_val)
    X_test = preprocessor.transform(X_test)

    # Encode the labels 
    le_label = LabelEncoder()
    y_train = le_label.fit_transform(y_train)
    y_val = le_label.transform(y_val)
    y_test = le_label.transform(y_test)

    # Convert to PyTorch tensors
    X_train = torch.FloatTensor(X_train)
    X_test = torch.FloatTensor(X_test)
    X_val = torch.FloatTensor(X_val)

    y_train = torch.FloatTensor(y_train)  
    y_val = torch.FloatTensor(y_val)      
    y_test = torch.LongTensor(y_test) 
   
    return X_train, X_test, X_val, y_train, y_test, y_val, preprocessor, le_label

In [10]:
X_train, X_test, X_val, y_train, y_test, y_val, preprocessor, le_label = prepare_data(X, y_binary)
    
print(f"\nDataset shape:")
print(f"Training set: {X_train.shape}")
print(f"Test set: {X_test.shape}")
print(f"Validation set: {X_val.shape}")


Dataset shape:
Training set: torch.Size([11003044, 74])
Test set: torch.Size([962514, 74])
Validation set: torch.Size([962514, 74])


In [11]:
label_mapping = dict(zip(le_label.classes_, le_label.transform(le_label.classes_)))
print(label_mapping)

{'Attack': np.int64(0), 'Benign': np.int64(1)}


In [12]:
# Define column groups
categorical_cols = [
        'Protocol',
        'Fwd PSH Flags','Fwd URG Flags',
        'FIN Flag Cnt','SYN Flag Cnt','RST Flag Cnt','ACK Flag Cnt',
        'URG Flag Cnt','PSH Flag Cnt','ECE Flag Cnt','CWE Flag Count'
        ]
    
numerical_cols = [c for c in X.columns if c not in categorical_cols]

# Get column names after preprocessing
ohe_feature_names = preprocessor.named_transformers_['cat'].get_feature_names_out(categorical_cols)
num_feature_names = numerical_cols
all_feature_names = np.concatenate([ohe_feature_names, num_feature_names])

In [13]:
train_dataset = TensorDataset(X_train, y_train)
val_dataset = TensorDataset(X_val, y_val)
test_dataset = TensorDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=2048, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=2048, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=2048, shuffle=False)

## Train Baseline Model

In [14]:
class DDoSDetector(nn.Module):
    def __init__(self, num_classes=1):
        super().__init__()

        layers = []
        in_channels = 1

        # 10 Conv1D layers
        for _ in range(10):
            layers.append(nn.Conv1d(in_channels, 108, kernel_size=5, padding=2))
            layers.append(nn.BatchNorm1d(108))
            layers.append(nn.ReLU())
            in_channels = 108

        self.cnn_backbone = nn.Sequential(*layers)
        self.global_pool = nn.AdaptiveAvgPool1d(1)
        self.classifier = nn.Linear(108, num_classes)

    def forward(self, x):
        # x: (batch_size, num_features)
        x = x.unsqueeze(1)                 # (B, 1, num_features)
        x = self.cnn_backbone(x)            # (B, 108, L)
        x = self.global_pool(x).squeeze(-1) # (B, 108)
        return self.classifier(x)            # (B, 1)
    
# Initialize model
model = DDoSDetector(num_classes=1)

In [15]:
if torch.cuda.is_available():
    model = model.to('cuda')

summary(model, input_size=(X_train.shape[1],))

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv1d-1              [-1, 108, 74]             648
       BatchNorm1d-2              [-1, 108, 74]             216
              ReLU-3              [-1, 108, 74]               0
            Conv1d-4              [-1, 108, 74]          58,428
       BatchNorm1d-5              [-1, 108, 74]             216
              ReLU-6              [-1, 108, 74]               0
            Conv1d-7              [-1, 108, 74]          58,428
       BatchNorm1d-8              [-1, 108, 74]             216
              ReLU-9              [-1, 108, 74]               0
           Conv1d-10              [-1, 108, 74]          58,428
      BatchNorm1d-11              [-1, 108, 74]             216
             ReLU-12              [-1, 108, 74]               0
           Conv1d-13              [-1, 108, 74]          58,428
      BatchNorm1d-14              [-1, 

In [ ]:
# Hyperparameters
num_classes = 1  
learning_rate = 1e-3
l2_lambda = 1e-5
l1_lambda = 1e-6
epochs = 5

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = DDoSDetector(num_classes=num_classes).to(device)

# Loss and Optimizer
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=l2_lambda)

# Training and Validation Loop
for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    
    # Wrap the loader in tqdm for progress bar
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}")
    
    for inputs, labels in pbar:
        inputs, labels = inputs.to(device, non_blocking=True), labels.to(device, non_blocking=True)

        optimizer.zero_grad()
        outputs = model(inputs).squeeze(1)  # Now (batch,) shape
        loss = criterion(outputs, labels.float())  

        # L1 Regularization
        l1_reg = torch.tensor(0., device=device)
        for p in model.parameters():
            l1_reg += p.abs().sum() 
        
        loss = loss + l1_lambda * l1_reg

        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        
        # Update the progress bar with the current loss every batch
        pbar.set_postfix({'loss': f"{loss.item():.4f}"})

    avg_train_loss = running_loss / len(train_loader)

    # --- VALIDATION PHASE---
    model.eval()
    val_running_loss = 0.0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            outputs = model(inputs).squeeze(1)
            # Calculate validation loss
            v_loss = criterion(outputs, labels.float())  # ← ADDED .float()
            val_running_loss += v_loss.item()

            v_probs = torch.sigmoid(outputs)
            v_predicted = (v_probs > 0.5).long()

            all_preds.extend(v_predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    # Calculate metrics
    avg_val_loss = val_running_loss / len(val_loader)
    val_accuracy = accuracy_score(all_labels, all_preds) * 100

    print(f"Epoch {epoch+1}: Train Loss {avg_train_loss:.4f}, Val Loss {avg_val_loss:.4f}, Val Acc: {val_accuracy:.2f}%")


# Save the baseline model weights
model_path = 'baseline_nids_cnn.pth'
torch.save(model.state_dict(), model_path)
print(f"Model baseline saved to {model_path}")

# FINAL TEST PHASE
model.eval()

# Lists to store all predictions and actual labels
all_preds = []
all_labels = []

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        outputs = model(inputs).squeeze(1)
        probs = torch.sigmoid(outputs)
        predictions = (probs>0.5).long()

        all_preds.extend(predictions.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

test_accuracy = accuracy_score(all_labels, all_preds) * 100
test_precision = precision_score(all_labels, all_preds)
test_recall = recall_score(all_labels, all_preds)
test_f1 = f1_score(all_labels, all_preds)
report = classification_report(all_labels, all_preds, target_names=['Attack', 'Benign'])


print("\n--- TEST METRICS ---")
print(f"Accuracy: {test_accuracy:.2f}%")
print(f"Precision: {test_precision:.4f}")
print(f"Recall: {test_recall:.4f}")
print(f"F1 Score: {test_f1:.4f}")
print("\nClassification Report:")
print(report)

Epoch 1:   0%|          | 0/5373 [00:00<?, ?it/s]

Epoch 1: 100%|██████████| 5373/5373 [03:59<00:00, 22.44it/s, loss=0.0855]


Epoch 1: Train Loss 0.1275, Val Loss 0.0867, Val Acc: 97.65%


Epoch 2: 100%|██████████| 5373/5373 [04:00<00:00, 22.32it/s, loss=0.1347]


Epoch 2: Train Loss 0.1162, Val Loss 0.0907, Val Acc: 97.68%


...

Epoch 3: Train Loss 0.1078, Val Loss 0.0838, Val Acc: 97.72%


Epoch 4: 100%|██████████| 5373/5373 [04:00<00:00, 22.38it/s, loss=0.1113]


Epoch 4: Train Loss 0.1068, Val Loss 0.0846, Val Acc: 97.68%


Epoch 5: 100%|██████████| 5373/5373 [03:59<00:00, 22.45it/s, loss=0.1022]


Epoch 5: Train Loss 0.1038, Val Loss 0.0806, Val Acc: 98.06%
Model baseline saved to baseline_nids_cnn.pth

--- TEST METRICS ---
Accuracy: 98.06%
Precision: 0.9756
Recall: 0.9978
F1 Score: 0.9865


## Generate Perturbation Set

### FGSM 

In [15]:
# Wrapper to convert binary classifier output to 2-class logits for foolbox
class DDoSDetectorBinaryWrapper(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model
    
    def forward(self, x):
        logit = self.model(x)
        zeros = torch.zeros_like(logit)
        return torch.cat([zeros, logit], dim=1)

In [20]:
def fgsm_attack_foolbox(model, X_train, 
                        y_train, 
                        epsilon_values=[0.01, 0.05, 0.1, 0.2, 0.3], 
                        batch_size=512, feature_indices=None,
                        mape_treshold=20.0):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)
    model.eval()
    
    # Wrap model to convert single logit to 2-class logits
    wrapped_model = DDoSDetectorBinaryWrapper(model).to(device)
    wrapped_model.eval()
    
    # Create foolbox model
    fmodel = fb.PyTorchModel(wrapped_model, bounds=(X_train.min().item(), X_train.max().item()))
    
    # Initialize FGSM attack
    attack = FGSM()
    
    results = {}
    samples_dict = {}
    
    for epsilon in epsilon_values:
        print(f"\n{'='*50}")
        print(f"Testing FGSM with epsilon = {epsilon}")
        print(f"{'='*50}")
        
        # Process in batches
        all_original_preds = []
        all_adv_preds = []
        all_perturbations = []
        all_original_probs = []
        all_adv_probs = []
        all_mape_constraint = []

        all_successful_samples = []
        
        num_batches = (len(X_train) + batch_size - 1) // batch_size
        
        for batch_idx in range(num_batches):
            start_idx = batch_idx * batch_size
            end_idx = min((batch_idx + 1) * batch_size, len(X_train))
            
            X_batch = X_train[start_idx:end_idx].to(device)
            y_batch = y_train[start_idx:end_idx].to(device)
            
            # Get original predictions
            with torch.no_grad():
                original_outputs = model(X_batch)
                original_probs = torch.sigmoid(original_outputs).squeeze()
                original_preds = (original_probs > 0.5).long()
            
            # Perform FGSM attack
            _, adversarial_batch, _ = attack(
                fmodel, 
                X_batch, 
                y_batch, 
                epsilons=epsilon
            )
            
            # Define a mask to only allow perturbations on the specified feature indices
            mask = torch.zeros_like(X_batch)
            mask[:, feature_indices] = 1.0

            # Apply mask to adversarial batch
            adversarial_batch = X_batch + ((adversarial_batch - X_batch) * mask)
            adversarial_batch = torch.clamp(adversarial_batch, X_train.min().item(), X_train.max().item())

            # Get adversarial predictions
            with torch.no_grad():
                adv_outputs = model(adversarial_batch)
                adv_probs = torch.sigmoid(adv_outputs).squeeze()
                adv_preds = (adv_probs > 0.5).long()

            # MAPE Constraint
            nonzero_mask = X_batch.abs() > 1e-2  # ignore near-zero features
            per_feature_mape = torch.abs((adversarial_batch - X_batch) / (X_batch.abs() + 1e-8)) * 100.0
            mape_constraint_mask = ((per_feature_mape <= mape_treshold) | ~nonzero_mask).all(dim=1)

            # per_feature_mape = torch.abs((adversarial_batch - X_batch) / (torch.abs(X_batch))) * 100.0
            # mape_constraint_mask = (per_feature_mape <= mape_treshold).all(dim=1)
            all_mape_constraint.append(mape_constraint_mask.cpu())  # store bool mask, not raw MAPE

            correctly_classified_mask = (original_preds == y_batch)
            successful_attacks = (adv_preds != y_batch) & correctly_classified_mask & mape_constraint_mask
            all_successful_samples.append((X_batch[successful_attacks].cpu(), adversarial_batch[successful_attacks].cpu()))
            
            # Calculate perturbation
            perturbation = (adversarial_batch - X_batch).abs()
            
            # Store results (move to CPU to save GPU memory)
            all_original_preds.append(original_preds.cpu())
            all_adv_preds.append(adv_preds.cpu())
            all_perturbations.append(perturbation.cpu())
            all_original_probs.append(original_probs.cpu())
            all_adv_probs.append(adv_probs.cpu())
            
            # Clear GPU cache
            del X_batch, y_batch, original_outputs, adv_outputs, adversarial_batch, perturbation
            torch.cuda.empty_cache()
            
            if (batch_idx + 1) % 10 == 0:
                print(f"Processed {end_idx}/{len(X_train)} samples...")

        if all_successful_samples:
            orig_tensors, adv_tensors = zip(*all_successful_samples)
            samples_dict[epsilon] = {
                'original': torch.cat(orig_tensors),
                'adversarial': torch.cat(adv_tensors)
                }
        else:
            samples_dict[epsilon] = {
                'original': torch.empty(0, X_train.shape[1]),
                'adversarial': torch.empty(0, X_train.shape[1])
                }
        
        # Concatenate all results
        original_preds = torch.cat(all_original_preds)
        adv_preds = torch.cat(all_adv_preds)
        all_perturbations = torch.cat(all_perturbations)
        original_probs = torch.cat(all_original_probs)
        adv_probs = torch.cat(all_adv_probs)
        y_train_cpu = y_train.cpu()
        
        # Calculate metrics
        original_accuracy = (original_preds == y_train_cpu).float().mean().item()
        adversarial_accuracy = (adv_preds == y_train_cpu).float().mean().item()
        
        # Calculate ASR
        correctly_classified = (original_preds == y_train_cpu)
        mape_constraint_global = torch.cat(all_mape_constraint)
        successful_attacks = (adv_preds != y_train_cpu) & correctly_classified & mape_constraint_global
        asr = successful_attacks.float().sum().item() / correctly_classified.float().sum().item()
        
        # Perturbation metrics
        avg_perturbation = all_perturbations.mean().item()
        max_perturbation = all_perturbations.max().item()
        l2_perturbation = torch.norm(all_perturbations, p=2, dim=1).mean().item()
        
        # Confidence drop
        confidence_drop = (original_probs - adv_probs).abs().mean().item()
        
        results[epsilon] = {
            'original_accuracy': original_accuracy,
            'adversarial_accuracy': adversarial_accuracy,
            'asr': asr,
            'mape_constraint_percentage': mape_constraint_global.float().mean().item() * 100.0,
            'avg_perturbation': avg_perturbation,
            'max_perturbation': max_perturbation,
            'l2_perturbation': l2_perturbation,
            'confidence_drop': confidence_drop,
            'num_samples': len(X_train),
            'num_correctly_classified': correctly_classified.sum().item(),
            'num_successful_attacks': successful_attacks.sum().item()
        }
        
        # Print results
        print(f"Original Accuracy: {original_accuracy*100:.2f}%")
        print(f"Adversarial Accuracy: {adversarial_accuracy*100:.2f}%")
        print(f"MAPE Constraint Satisfied: {mape_constraint_global.float().mean().item() * 100.0:.2f}% of samples")
        print(f"Attack Success Rate (ASR): {asr*100:.2f}%")
        print(f"Average L∞ Perturbation: {avg_perturbation:.6f}")
        print(f"Average L2 Perturbation: {l2_perturbation:.6f}")
        print(f"Max Perturbation: {max_perturbation:.6f}")
        print(f"Average Confidence Drop: {confidence_drop:.4f}")
        print(f"Successfully Attacked: {successful_attacks.sum().item()}/{correctly_classified.sum().item()}")
    
    return results, samples_dict

In [22]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Get baseline model
baseline_model = DDoSDetector(num_classes=1).to(device)
baseline_model.load_state_dict(torch.load("baseline_nids_cnn.pth", map_location=device))

<All keys matched successfully>

In [23]:
features_to_peturb = ['Flow Duration', 'Flow IAT Mean', 'Flow IAT Max', 'Flow IAT Min', 'Flow IAT Std',
                      'Fwd IAT Tot', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min',
                      'Bwd IAT Tot', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min',
                      'Active Mean', 'Active Std', 'Active Max', 'Active Min',
                      'Idle Mean', 'Idle Std', 'Idle Max', 'Idle Min'   
]
feature_indices = [numerical_cols.index(feature) for feature in features_to_peturb]

# Shuffle and Split indices
indices = torch.randperm(len(X_train))
split_point = len(X_train) // 2

clean_idx = indices[:split_point]
perturb_idx = indices[split_point:]

# Get the halves
X_clean, y_clean = X_train[clean_idx], y_train[clean_idx]
X_to_perturb, y_to_perturb = X_train[perturb_idx], y_train[perturb_idx]

# Generate Adversarial Samples 
is_malicious = (y_to_perturb == 0)

results, sample_dict = fgsm_attack_foolbox(baseline_model, X_to_perturb[is_malicious],
                                            y_to_perturb[is_malicious].long(), 
                                            epsilon_values=[0.05, 0.07, 0.1], 
                                            batch_size=4096, 
                                            feature_indices=feature_indices)


Testing FGSM with epsilon = 0.05
Processed 40960/2752596 samples...
Processed 81920/2752596 samples...
Processed 122880/2752596 samples...
Processed 163840/2752596 samples...
Processed 204800/2752596 samples...
Processed 245760/2752596 samples...
Processed 286720/2752596 samples...
Processed 327680/2752596 samples...
Processed 368640/2752596 samples...
Processed 409600/2752596 samples...
Processed 450560/2752596 samples...
Processed 491520/2752596 samples...
Processed 532480/2752596 samples...
Processed 573440/2752596 samples...
Processed 614400/2752596 samples...
Processed 655360/2752596 samples...
Processed 696320/2752596 samples...
Processed 737280/2752596 samples...
Processed 778240/2752596 samples...
Processed 819200/2752596 samples...
Processed 860160/2752596 samples...
Processed 901120/2752596 samples...
Processed 942080/2752596 samples...
Processed 983040/2752596 samples...
Processed 1024000/2752596 samples...
Processed 1064960/2752596 samples...
Processed 1105920/2752596 samp

In [24]:
X_clean.shape, y_clean.shape, X_to_perturb.shape, y_to_perturb.shape, X_to_perturb[is_malicious].shape, y_to_perturb[is_malicious].shape, X_to_perturb[~is_malicious].shape, y_to_perturb[~is_malicious].shape

(torch.Size([5501522, 74]),
 torch.Size([5501522]),
 torch.Size([5501522, 74]),
 torch.Size([5501522]),
 torch.Size([2752596, 74]),
 torch.Size([2752596]),
 torch.Size([2748926, 74]),
 torch.Size([2748926]))

In [27]:
# # To confirm it works
# eps = 0.05 
# original_data = sample_dict[eps]['original']
# adversarial_data = sample_dict[eps]['adversarial']

# # Get model predictions for both
# with torch.no_grad():
#     orig_pred = (torch.sigmoid(model(original_data.to(device))) > 0.5).long().cpu()
#     adv_pred = (torch.sigmoid(model(adversarial_data.to(device))) > 0.5).long().cpu()

# print(f"Original predictions (should be 0): {orig_pred[:5].numpy()}")
# print(f"Adversarial predictions (should be 1): {adv_pred[:5].numpy()}")

In [28]:
results_fgsm = pd.DataFrame.from_dict(results, orient='index')
results_fgsm.index.name = 'epsilon'
results_fgsm.reset_index(inplace=True)
results_fgsm.to_csv('fgsm_attack_results.csv', index=False)
results_fgsm

,epsilon,original_accuracy,adversarial_accuracy,asr,mape_constraint_percentage,avg_perturbation,max_perturbation,l2_perturbation,confidence_drop,num_samples,num_correctly_classified,num_successful_attacks
0,0.05,0.947707,0.672022,0.263307,76.808548,0.009180,0.05,0.183303,0.284792,2752596,2608654,686876
1,0.07,0.947707,0.524991,0.394896,74.579960,0.012809,0.07,0.256252,0.363826,2752596,2608654,1030148
2,0.10,0.947707,0.474293,0.397309,72.115958,0.018226,0.10,0.365338,0.494956,2752596,2608654,1036443


In [29]:
# Store all original and adversarial samples across all epsilons
paired_list = []

for v in sample_dict.values():
    if v['adversarial'].shape[0] > 0:
        # index 0: original, index 1: adversarial
        pairs = torch.stack([v['original'], v['adversarial']], dim=1)
        paired_list.append(pairs)

X_combined = torch.cat(paired_list, dim=0)
y_adv_all = torch.zeros(X_combined.shape[0])

# Store FGSM original and adversarial dataset
torch.save((X_combined, y_adv_all), 'fgsm_orig_and_adv_dataset.pt')
print("FGSM original and adversarial dataset saved as 'fgsm_orig_and_adv_dataset.pt'")

FGSM original and adversarial dataset saved as 'fgsm_orig_and_adv_dataset.pt'


In [30]:
# Collect all adversarial tensors across all epsilon keys
all_adv_list = [v['adversarial'] for v in sample_dict.values() if v['adversarial'].shape[0] > 0]

# Concatenate them into one big "Hacker Pack"
X_adv_all = torch.cat(all_adv_list, dim=0)

# Create the labels (Attack is 0)
y_adv_all = torch.zeros(X_adv_all.shape[0])

In [31]:
X_adv_all.shape, y_adv_all.shape

(torch.Size([2753467, 74]), torch.Size([2753467]))

In [32]:
# Merge: Clean half + All Adv samples + The Benign half of the perturbation set
X_train_new = torch.cat([X_clean, X_adv_all, X_to_perturb[~is_malicious]], dim=0)
y_train_new = torch.cat([y_clean, y_adv_all, y_to_perturb[~is_malicious]], dim=0)

# Shuffle the new training set
shuffled_idx = torch.randperm(len(X_train_new))
X_train_new = X_train_new[shuffled_idx]
y_train_new = y_train_new[shuffled_idx]

In [33]:
X_train_new.shape, y_train_new.shape

(torch.Size([11003915, 74]), torch.Size([11003915]))

In [34]:
# Store the FGSM augmented dataset
torch.save((X_train_new, y_train_new), 'fgsm_augmented_dataset.pt')
print("FGSM augmented dataset saved as 'fgsm_augmented_dataset.pt'")

FGSM augmented dataset saved as 'fgsm_augmented_dataset.pt'


### PGD

In [21]:
# Wrapper to convert binary classifier output to 2-class logits for foolbox
class DDoSDetectorBinaryWrapper(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model
    
    def forward(self, x):
        logit = self.model(x)  # (B, 1)
        negative_logit = -logit
        return torch.cat([negative_logit, logit], dim=1)  # (B, 2)

In [ ]:
def pgd_attack_foolbox(model, X_train, 
                        y_train, 
                        epsilon_values=[0.05, 0.07, 0.1], 
                        batch_size=512, feature_indices=None,
                        mape_treshold=20.0):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)
    model.eval()
    
    # Wrap model to convert single logit to 2-class logits
    wrapped_model = DDoSDetectorBinaryWrapper(model).to(device)
    wrapped_model.eval()
    
    # Create foolbox model
    fmodel = fb.PyTorchModel(wrapped_model, bounds=(X_train.min().item(), X_train.max().item()))
    
    # Initialize PGD attack
    attack = PGD(steps=40, rel_stepsize=0.1)
    
    
    results = {}
    samples_dict = {}
    
    for epsilon in epsilon_values:
        print(f"\n{'='*50}")
        print(f"Testing PGD with epsilon = {epsilon}")
        print(f"{'='*50}")
        
        # Process in batches
        all_original_preds = []
        all_adv_preds = []
        all_perturbations = []
        all_original_probs = []
        all_adv_probs = []
        all_mape_constraint = []

        all_successful_samples = []
        
        num_batches = (len(X_train) + batch_size - 1) // batch_size
        
        for batch_idx in range(num_batches):
            start_idx = batch_idx * batch_size
            end_idx = min((batch_idx + 1) * batch_size, len(X_train))
            
            X_batch = X_train[start_idx:end_idx].to(device)
            y_batch = y_train[start_idx:end_idx].to(device)
            
            # Get original predictions
            with torch.no_grad():
                original_outputs = model(X_batch)
                original_probs = torch.sigmoid(original_outputs).squeeze()
                original_preds = (original_probs > 0.5).long()
            
            # Perform PGD attack
            _, adversarial_batch, _ = attack(
                fmodel, 
                X_batch, 
                y_batch, 
                epsilons=epsilon
            )
            
            # Define a mask to only allow perturbations on the specified feature indices
            mask = torch.zeros_like(X_batch)
            mask[:, feature_indices] = 1.0

            # Apply mask to adversarial batch
            adversarial_batch = X_batch + ((adversarial_batch - X_batch) * mask)
            adversarial_batch = torch.clamp(adversarial_batch, X_train.min().item(), X_train.max().item())

            # Get adversarial predictions
            with torch.no_grad():
                adv_outputs = model(adversarial_batch)
                adv_probs = torch.sigmoid(adv_outputs).squeeze()
                adv_preds = (adv_probs > 0.5).long()

            # MAPE Constraint
            nonzero_mask = X_batch.abs() > 1e-2  # ignore near-zero features
            per_feature_mape = torch.abs((adversarial_batch - X_batch) / (X_batch.abs() + 1e-8)) * 100.0
            mape_constraint_mask = ((per_feature_mape <= mape_treshold) | ~nonzero_mask).all(dim=1)

            # per_feature_mape = torch.abs((adversarial_batch - X_batch) / (torch.abs(X_batch))) * 100.0
            # mape_constraint_mask = (per_feature_mape <= mape_treshold).all(dim=1)
            all_mape_constraint.append(mape_constraint_mask.cpu())  # store bool mask, not raw MAPE

            correctly_classified_mask = (original_preds == y_batch)
            successful_attacks = (adv_preds != y_batch) & correctly_classified_mask & mape_constraint_mask
            all_successful_samples.append((X_batch[successful_attacks].cpu(), adversarial_batch[successful_attacks].cpu()))
            
            # Calculate perturbation
            perturbation = (adversarial_batch - X_batch).abs()
            
            # Store results (move to CPU to save GPU memory)
            all_original_preds.append(original_preds.cpu())
            all_adv_preds.append(adv_preds.cpu())
            all_perturbations.append(perturbation.cpu())
            all_original_probs.append(original_probs.cpu())
            all_adv_probs.append(adv_probs.cpu())
            
            # Clear GPU cache
            del X_batch, y_batch, original_outputs, adv_outputs, adversarial_batch, perturbation
            torch.cuda.empty_cache()
            
            if (batch_idx + 1) % 10 == 0:
                print(f"Processed {end_idx}/{len(X_train)} samples...")

        if all_successful_samples:
            orig_tensors, adv_tensors = zip(*all_successful_samples)
            samples_dict[epsilon] = {
                'original': torch.cat(orig_tensors),
                'adversarial': torch.cat(adv_tensors)
                }
        else:
            samples_dict[epsilon] = {
                'original': torch.empty(0, X_train.shape[1]),
                'adversarial': torch.empty(0, X_train.shape[1])
                }
        
        # Concatenate all results
        original_preds = torch.cat(all_original_preds)
        adv_preds = torch.cat(all_adv_preds)
        all_perturbations = torch.cat(all_perturbations)
        original_probs = torch.cat(all_original_probs)
        adv_probs = torch.cat(all_adv_probs)
        y_train_cpu = y_train.cpu()
        
        # Calculate metrics
        original_accuracy = (original_preds == y_train_cpu).float().mean().item()
        adversarial_accuracy = (adv_preds == y_train_cpu).float().mean().item()
        
        # Calculate ASR
        correctly_classified = (original_preds == y_train_cpu)
        mape_constraint_global = torch.cat(all_mape_constraint)
        successful_attacks = (adv_preds != y_train_cpu) & correctly_classified & mape_constraint_global
        asr = successful_attacks.float().sum().item() / correctly_classified.float().sum().item()
        
        # Perturbation metrics
        avg_perturbation = all_perturbations.mean().item()
        max_perturbation = all_perturbations.max().item()
        l2_perturbation = torch.norm(all_perturbations, p=2, dim=1).mean().item()
        
        # Confidence drop
        confidence_drop = (original_probs - adv_probs).abs().mean().item()
        
        results[epsilon] = {
            'original_accuracy': original_accuracy,
            'adversarial_accuracy': adversarial_accuracy,
            'asr': asr,
            'mape_constraint_percentage': mape_constraint_global.float().mean().item() * 100.0,
            'avg_perturbation': avg_perturbation,
            'max_perturbation': max_perturbation,
            'l2_perturbation': l2_perturbation,
            'confidence_drop': confidence_drop,
            'num_samples': len(X_train),
            'num_correctly_classified': correctly_classified.sum().item(),
            'num_successful_attacks': successful_attacks.sum().item()
        }
        
        # Print results
        print(f"Original Accuracy: {original_accuracy*100:.2f}%")
        print(f"Adversarial Accuracy: {adversarial_accuracy*100:.2f}%")
        print(f"MAPE Constraint Satisfied: {mape_constraint_global.float().mean().item() * 100.0:.2f}% of samples")
        print(f"Attack Success Rate (ASR): {asr*100:.2f}%")
        print(f"Average L∞ Perturbation: {avg_perturbation:.6f}")
        print(f"Average L2 Perturbation: {l2_perturbation:.6f}")
        print(f"Max Perturbation: {max_perturbation:.6f}")
        print(f"Average Confidence Drop: {confidence_drop:.4f}")
        print(f"Successfully Attacked: {successful_attacks.sum().item()}/{correctly_classified.sum().item()}")
    
    return results, samples_dict

In [19]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Get baseline model
baseline_model = DDoSDetector(num_classes=1).to(device)
baseline_model.load_state_dict(torch.load("baseline_nids_cnn.pth", map_location=device))

<All keys matched successfully>

In [21]:
features_to_peturb = ['Flow Duration', 'Flow IAT Mean', 'Flow IAT Max', 'Flow IAT Min', 'Flow IAT Std',
                      'Fwd IAT Tot', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min',
                      'Bwd IAT Tot', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min',
                      'Active Mean', 'Active Std', 'Active Max', 'Active Min',
                      'Idle Mean', 'Idle Std', 'Idle Max', 'Idle Min'   
]
feature_indices = [numerical_cols.index(feature) for feature in features_to_peturb]

# Shuffle and Split indices
indices = torch.randperm(len(X_train))
split_point = len(X_train) // 2

clean_idx = indices[:split_point]
perturb_idx = indices[split_point:]

# Get the halves
X_clean, y_clean = X_train[clean_idx], y_train[clean_idx]
X_to_perturb, y_to_perturb = X_train[perturb_idx], y_train[perturb_idx]

# Generate Adversarial Samples 
is_malicious = (y_to_perturb == 0)

results, sample_dict = pgd_attack_foolbox(baseline_model, X_to_perturb[is_malicious],
                                            y_to_perturb[is_malicious].long(), 
                                            epsilon_values=[0.05, 0.07, 0.1], 
                                            batch_size=8192, 
                                            feature_indices=feature_indices)


Testing PGD with epsilon = 0.05
Processed 81920/2751934 samples...
Processed 163840/2751934 samples...
Processed 245760/2751934 samples...
Processed 327680/2751934 samples...
Processed 409600/2751934 samples...
Processed 491520/2751934 samples...
Processed 573440/2751934 samples...
Processed 655360/2751934 samples...
Processed 737280/2751934 samples...
Processed 819200/2751934 samples...
Processed 901120/2751934 samples...
Processed 983040/2751934 samples...
Processed 1064960/2751934 samples...
Processed 1146880/2751934 samples...
Processed 1228800/2751934 samples...
Processed 1310720/2751934 samples...
Processed 1392640/2751934 samples...
Processed 1474560/2751934 samples...
Processed 1556480/2751934 samples...
Processed 1638400/2751934 samples...
Processed 1720320/2751934 samples...
Processed 1802240/2751934 samples...
Processed 1884160/2751934 samples...
Processed 1966080/2751934 samples...
Processed 2048000/2751934 samples...
Processed 2129920/2751934 samples...
Processed 2211840/

In [22]:
results_pgd = pd.DataFrame.from_dict(results, orient='index')
results_pgd.index.name = 'epsilon'
results_pgd.reset_index(inplace=True)
results_pgd.to_csv('pgd_attack_results.csv', index=False)
results_pgd

,epsilon,original_accuracy,adversarial_accuracy,asr,mape_constraint_percentage,avg_perturbation,max_perturbation,l2_perturbation,confidence_drop,num_samples,num_correctly_classified,num_successful_attacks
0,0.05,0.947788,0.800409,0.138013,77.272236,0.007747,0.05,0.151669,0.161530,2751934,2608249,359972
1,0.07,0.947788,0.753329,0.164741,75.553268,0.010529,0.07,0.206081,0.205099,2751934,2608249,429685
2,0.10,0.947788,0.656547,0.228255,72.784084,0.014563,0.10,0.285798,0.299447,2751934,2608249,595345


In [23]:
# Store all original and adversarial samples across all epsilons
paired_list = []

for v in sample_dict.values():
    if v['adversarial'].shape[0] > 0:
        # index 0: original, index 1: adversarial
        pairs = torch.stack([v['original'], v['adversarial']], dim=1)
        paired_list.append(pairs)

X_combined = torch.cat(paired_list, dim=0)
y_adv_all = torch.zeros(X_combined.shape[0])

# Store PGD original and adversarial dataset
torch.save((X_combined, y_adv_all), 'pgd_orig_and_adv_dataset.pt')
print("PGD original and adversarial dataset saved as 'pgd_orig_and_adv_dataset.pt'")

PGD original and adversarial dataset saved as 'pgd_orig_and_adv_dataset.pt'


In [24]:
# Collect all adversarial tensors across all epsilon keys
all_adv_list = [v['adversarial'] for v in sample_dict.values() if v['adversarial'].shape[0] > 0]

# Concatenate them into one big "Hacker Pack"
X_adv_all = torch.cat(all_adv_list, dim=0)

# Create the labels (Attack is 0)
y_adv_all = torch.zeros(X_adv_all.shape[0])

In [25]:
X_adv_all.shape, y_adv_all.shape

(torch.Size([1385002, 74]), torch.Size([1385002]))

In [26]:
# Merge: Clean half + All Adv samples + The Benign half of the perturbation set
X_train_new = torch.cat([X_clean, X_adv_all, X_to_perturb[~is_malicious]], dim=0)
y_train_new = torch.cat([y_clean, y_adv_all, y_to_perturb[~is_malicious]], dim=0)

# Shuffle the new training set
shuffled_idx = torch.randperm(len(X_train_new))
X_train_new = X_train_new[shuffled_idx]
y_train_new = y_train_new[shuffled_idx]

In [27]:
X_train_new.shape, y_train_new.shape

(torch.Size([9636112, 74]), torch.Size([9636112]))

In [28]:
# Store the PGD augmented dataset
torch.save((X_train_new, y_train_new), 'pgd_augmented_dataset.pt')
print("PGD augmented dataset saved as 'pgd_augmented_dataset.pt'")

PGD augmented dataset saved as 'pgd_augmented_dataset.pt'


### I-FGSM

In [15]:
# Wrapper to convert binary classifier output to 2-class logits for foolbox
class DDoSDetectorBinaryWrapper(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model
    
    def forward(self, x):
        logit = self.model(x)  # (B, 1)
        negative_logit = -logit
        return torch.cat([negative_logit, logit], dim=1)  # (B, 2)

In [16]:
def ifgsm_attack_foolbox(model, X_train, 
                        y_train, 
                        epsilon_values=[0.05, 0.07, 0.1], 
                        batch_size=512, feature_indices=None,
                        mape_treshold=20.0):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)
    model.eval()
    
    # Wrap model to convert single logit to 2-class logits
    wrapped_model = DDoSDetectorBinaryWrapper(model).to(device)
    wrapped_model.eval()
    
    # Create foolbox model
    fmodel = fb.PyTorchModel(wrapped_model, bounds=(X_train.min().item(), X_train.max().item()))
    
    # Initialize IFGSM attack
    attack = LinfBasicIterativeAttack(steps=40, rel_stepsize=0.1)
    
    results = {}
    samples_dict = {}
    
    for epsilon in epsilon_values:
        print(f"\n{'='*50}")
        print(f"Testing IFGSM with epsilon = {epsilon}")
        print(f"{'='*50}")
        
        # Process in batches
        all_original_preds = []
        all_adv_preds = []
        all_perturbations = []
        all_original_probs = []
        all_adv_probs = []
        all_mape_constraint = []

        all_successful_samples = []
        
        num_batches = (len(X_train) + batch_size - 1) // batch_size
        
        for batch_idx in range(num_batches):
            start_idx = batch_idx * batch_size
            end_idx = min((batch_idx + 1) * batch_size, len(X_train))
            
            X_batch = X_train[start_idx:end_idx].to(device)
            y_batch = y_train[start_idx:end_idx].to(device)
            
            # Get original predictions
            with torch.no_grad():
                original_outputs = model(X_batch)
                original_probs = torch.sigmoid(original_outputs).squeeze()
                original_preds = (original_probs > 0.5).long()
            
            # Perform IFGSM attack
            _, adversarial_batch, _ = attack(
                fmodel, 
                X_batch, 
                y_batch, 
                epsilons=epsilon
            )
            
            # Define a mask to only allow perturbations on the specified feature indices
            mask = torch.zeros_like(X_batch)
            mask[:, feature_indices] = 1.0

            # Apply mask to adversarial batch
            adversarial_batch = X_batch + ((adversarial_batch - X_batch) * mask)
            adversarial_batch = torch.clamp(adversarial_batch, X_train.min().item(), X_train.max().item())

            # Get adversarial predictions
            with torch.no_grad():
                adv_outputs = model(adversarial_batch)
                adv_probs = torch.sigmoid(adv_outputs).squeeze()
                adv_preds = (adv_probs > 0.5).long()

            # MAPE Constraint
            nonzero_mask = X_batch.abs() > 1e-2  # ignore near-zero features
            per_feature_mape = torch.abs((adversarial_batch - X_batch) / (X_batch.abs() + 1e-8)) * 100.0
            mape_constraint_mask = ((per_feature_mape <= mape_treshold) | ~nonzero_mask).all(dim=1)

            # per_feature_mape = torch.abs((adversarial_batch - X_batch) / (torch.abs(X_batch))) * 100.0
            # mape_constraint_mask = (per_feature_mape <= mape_treshold).all(dim=1)
            all_mape_constraint.append(mape_constraint_mask.cpu())  # store bool mask, not raw MAPE

            correctly_classified_mask = (original_preds == y_batch)
            successful_attacks = (adv_preds != y_batch) & correctly_classified_mask & mape_constraint_mask
            all_successful_samples.append((X_batch[successful_attacks].cpu(), adversarial_batch[successful_attacks].cpu()))
            
            # Calculate perturbation
            perturbation = (adversarial_batch - X_batch).abs()
            
            # Store results (move to CPU to save GPU memory)
            all_original_preds.append(original_preds.cpu())
            all_adv_preds.append(adv_preds.cpu())
            all_perturbations.append(perturbation.cpu())
            all_original_probs.append(original_probs.cpu())
            all_adv_probs.append(adv_probs.cpu())
            
            # Clear GPU cache
            del X_batch, y_batch, original_outputs, adv_outputs, adversarial_batch, perturbation
            torch.cuda.empty_cache()
            
            if (batch_idx + 1) % 10 == 0:
                print(f"Processed {end_idx}/{len(X_train)} samples...")

        if all_successful_samples:
            orig_tensors, adv_tensors = zip(*all_successful_samples)
            samples_dict[epsilon] = {
                'original': torch.cat(orig_tensors),
                'adversarial': torch.cat(adv_tensors)
                }
        else:
            samples_dict[epsilon] = {
                'original': torch.empty(0, X_train.shape[1]),
                'adversarial': torch.empty(0, X_train.shape[1])
                }
        
        # Concatenate all results
        original_preds = torch.cat(all_original_preds)
        adv_preds = torch.cat(all_adv_preds)
        all_perturbations = torch.cat(all_perturbations)
        original_probs = torch.cat(all_original_probs)
        adv_probs = torch.cat(all_adv_probs)
        y_train_cpu = y_train.cpu()
        
        # Calculate metrics
        original_accuracy = (original_preds == y_train_cpu).float().mean().item()
        adversarial_accuracy = (adv_preds == y_train_cpu).float().mean().item()
        
        # Calculate ASR
        correctly_classified = (original_preds == y_train_cpu)
        mape_constraint_global = torch.cat(all_mape_constraint)
        successful_attacks = (adv_preds != y_train_cpu) & correctly_classified & mape_constraint_global
        asr = successful_attacks.float().sum().item() / correctly_classified.float().sum().item()
        
        # Perturbation metrics
        avg_perturbation = all_perturbations.mean().item()
        max_perturbation = all_perturbations.max().item()
        l2_perturbation = torch.norm(all_perturbations, p=2, dim=1).mean().item()
        
        # Confidence drop
        confidence_drop = (original_probs - adv_probs).abs().mean().item()
        
        results[epsilon] = {
            'original_accuracy': original_accuracy,
            'adversarial_accuracy': adversarial_accuracy,
            'asr': asr,
            'mape_constraint_percentage': mape_constraint_global.float().mean().item() * 100.0,
            'avg_perturbation': avg_perturbation,
            'max_perturbation': max_perturbation,
            'l2_perturbation': l2_perturbation,
            'confidence_drop': confidence_drop,
            'num_samples': len(X_train),
            'num_correctly_classified': correctly_classified.sum().item(),
            'num_successful_attacks': successful_attacks.sum().item()
        }
        
        # Print results
        print(f"Original Accuracy: {original_accuracy*100:.2f}%")
        print(f"Adversarial Accuracy: {adversarial_accuracy*100:.2f}%")
        print(f"MAPE Constraint Satisfied: {mape_constraint_global.float().mean().item() * 100.0:.2f}% of samples")
        print(f"Attack Success Rate (ASR): {asr*100:.2f}%")
        print(f"Average L∞ Perturbation: {avg_perturbation:.6f}")
        print(f"Average L2 Perturbation: {l2_perturbation:.6f}")
        print(f"Max Perturbation: {max_perturbation:.6f}")
        print(f"Average Confidence Drop: {confidence_drop:.4f}")
        print(f"Successfully Attacked: {successful_attacks.sum().item()}/{correctly_classified.sum().item()}")
        
    
    return results, samples_dict

In [17]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Get baseline model
baseline_model = DDoSDetector(num_classes=1).to(device)
baseline_model.load_state_dict(torch.load("baseline_nids_cnn.pth", map_location=device))

<All keys matched successfully>

In [18]:
features_to_peturb = ['Flow Duration', 'Flow IAT Mean', 'Flow IAT Max', 'Flow IAT Min', 'Flow IAT Std',
                      'Fwd IAT Tot', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min',
                      'Bwd IAT Tot', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min',
                      'Active Mean', 'Active Std', 'Active Max', 'Active Min',
                      'Idle Mean', 'Idle Std', 'Idle Max', 'Idle Min'   
]
feature_indices = [numerical_cols.index(feature) for feature in features_to_peturb]

# Shuffle and Split indices
indices = torch.randperm(len(X_train))
split_point = len(X_train) // 2

clean_idx = indices[:split_point]
perturb_idx = indices[split_point:]

# Get the halves
X_clean, y_clean = X_train[clean_idx], y_train[clean_idx]
X_to_perturb, y_to_perturb = X_train[perturb_idx], y_train[perturb_idx]

# Generate Adversarial Samples 
is_malicious = (y_to_perturb == 0)

results, sample_dict = ifgsm_attack_foolbox(baseline_model, X_to_perturb[is_malicious],
                                            y_to_perturb[is_malicious].long(), 
                                            epsilon_values=[0.05, 0.07, 0.1], 
                                            batch_size=8192, 
                                            feature_indices=feature_indices)


Testing IFGSM with epsilon = 0.05
Processed 81920/2750035 samples...
Processed 163840/2750035 samples...
Processed 245760/2750035 samples...
Processed 327680/2750035 samples...
Processed 409600/2750035 samples...
Processed 491520/2750035 samples...
Processed 573440/2750035 samples...
Processed 655360/2750035 samples...
Processed 737280/2750035 samples...
Processed 819200/2750035 samples...
Processed 901120/2750035 samples...
Processed 983040/2750035 samples...
Processed 1064960/2750035 samples...
Processed 1146880/2750035 samples...
Processed 1228800/2750035 samples...
Processed 1310720/2750035 samples...
Processed 1392640/2750035 samples...
Processed 1474560/2750035 samples...
Processed 1556480/2750035 samples...
Processed 1638400/2750035 samples...
Processed 1720320/2750035 samples...
Processed 1802240/2750035 samples...
Processed 1884160/2750035 samples...
Processed 1966080/2750035 samples...
Processed 2048000/2750035 samples...
Processed 2129920/2750035 samples...
Processed 221184

In [19]:
results_ifgsm = pd.DataFrame.from_dict(results, orient='index')
results_ifgsm.index.name = 'epsilon'
results_ifgsm.reset_index(inplace=True)
results_ifgsm.to_csv('ifgsm_attack_results.csv', index=False)
results_ifgsm

,epsilon,original_accuracy,adversarial_accuracy,asr,mape_constraint_percentage,avg_perturbation,max_perturbation,l2_perturbation,confidence_drop,num_samples,num_correctly_classified,num_successful_attacks
0,0.05,0.947703,0.741796,0.203900,77.410543,0.007352,0.05,0.144931,0.195089,2750035,2606216,531408
1,0.07,0.947703,0.719158,0.210546,76.050597,0.010134,0.07,0.201346,0.228481,2750035,2606216,548729
2,0.10,0.947703,0.634796,0.279121,73.074162,0.013038,0.10,0.267432,0.279037,2750035,2606216,727450


In [20]:
# Store all original and adversarial samples across all epsilons
paired_list = []

for v in sample_dict.values():
    if v['adversarial'].shape[0] > 0:
        # index 0: original, index 1: adversarial
        pairs = torch.stack([v['original'], v['adversarial']], dim=1)
        paired_list.append(pairs)

X_combined = torch.cat(paired_list, dim=0)
y_adv_all = torch.zeros(X_combined.shape[0])

# Store IFGSM original and adversarial dataset
torch.save((X_combined, y_adv_all), 'ifgsm_orig_and_adv_dataset.pt')
print("IFGSM original and adversarial dataset saved as 'ifgsm_orig_and_adv_dataset.pt'")

IFGSM original and adversarial dataset saved as 'ifgsm_orig_and_adv_dataset.pt'


In [21]:
# Collect all adversarial tensors across all epsilon keys
all_adv_list = [v['adversarial'] for v in sample_dict.values() if v['adversarial'].shape[0] > 0]

# Concatenate them into one big "Hacker Pack"
X_adv_all = torch.cat(all_adv_list, dim=0)

# Create the labels (Attack is 0)
y_adv_all = torch.zeros(X_adv_all.shape[0])

In [22]:
X_adv_all.shape, y_adv_all.shape

(torch.Size([1807587, 74]), torch.Size([1807587]))

In [23]:
# Merge: Clean half + All Adv samples + The Benign half of the perturbation set
X_train_new = torch.cat([X_clean, X_adv_all, X_to_perturb[~is_malicious]], dim=0)
y_train_new = torch.cat([y_clean, y_adv_all, y_to_perturb[~is_malicious]], dim=0)

# Shuffle the new training set
shuffled_idx = torch.randperm(len(X_train_new))
X_train_new = X_train_new[shuffled_idx]
y_train_new = y_train_new[shuffled_idx]

In [24]:
X_train_new.shape, y_train_new.shape

(torch.Size([10060596, 74]), torch.Size([10060596]))

In [25]:
# Store the IFGSM augmented dataset
torch.save((X_train_new, y_train_new), 'ifgsm_augmented_dataset.pt')
print("IFGSM augmented dataset saved as 'ifgsm_augmented_dataset.pt'")

IFGSM augmented dataset saved as 'ifgsm_augmented_dataset.pt'


## Train Teacher Models

### FGSM Teacher Model 

In [22]:
# Hyperparameters
learning_rate = 1e-3
l2_lambda = 1e-5
l1_lambda = 1e-6
epochs = 10
mape_threshold = 20.0
features_to_peturb = ['Flow Duration', 'Flow IAT Mean', 'Flow IAT Max', 'Flow IAT Min', 'Flow IAT Std',
                      'Fwd IAT Tot', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min',
                      'Bwd IAT Tot', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min',
                      'Active Mean', 'Active Std', 'Active Max', 'Active Min',
                      'Idle Mean', 'Idle Std', 'Idle Max', 'Idle Min'   
]
feature_indices = [numerical_cols.index(feature) for feature in features_to_peturb]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Split dataset once before training
indices = torch.randperm(len(X_train))
split = len(X_train) // 2

clean_idx = indices[:split]
adv_idx = indices[split:]

X_clean, y_clean = X_train[clean_idx], y_train[clean_idx]
X_adv_pool, y_adv_pool = X_train[adv_idx], y_train[adv_idx]

# Create dataloaders
train_clean_loader = DataLoader(TensorDataset(X_clean, y_clean), batch_size=1024, shuffle=True)
train_adv_loader = DataLoader(TensorDataset(X_adv_pool, y_adv_pool), batch_size=1024, shuffle=True)

# Validation and test loaders (clean data only)
val_loader = DataLoader(TensorDataset(X_val, y_val), batch_size=1024, shuffle=False)
test_loader = DataLoader(TensorDataset(X_test, y_test), batch_size=1024, shuffle=False)

# Initialize model from baseline
model = DDoSDetector(num_classes=1).to(device)
model.load_state_dict(torch.load('baseline_nids_cnn.pth', map_location=device))

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=l2_lambda)

# attack = LinfBasicIterativeAttack(abs_stepsize=0.01, steps=10)
attack = PGD(rel_stepsize=0.02, steps=10, random_start=True)
# attack = FGSM()  

x_min = X_train.min().item()
x_max = X_train.max().item()

perturbation_mask = torch.zeros(X_train.shape[1], device=device)
perturbation_mask[feature_indices] = 1.0

model.eval()
wrapped = DDoSDetectorBinaryWrapper(model).to(device)
fmodel = fb.PyTorchModel(wrapped, bounds=(x_min, x_max))

for epoch in range(epochs):
    running_loss = 0.0

    pbar = tqdm(zip(train_clean_loader, train_adv_loader), 
                desc=f"Epoch {epoch+1}",
                total=min(len(train_clean_loader), len(train_adv_loader)))

    for (clean_inputs, clean_labels), (adv_inputs, adv_labels) in pbar:
        clean_inputs = clean_inputs.to(device)
        clean_labels = clean_labels.to(device)
        adv_inputs = adv_inputs.to(device)
        adv_labels = adv_labels.to(device)

        # INNER MAX — generate adversarial examples on the fly from adv pool
        model.eval()
        is_malicious = (adv_labels == 0)
        adv_inputs_perturbed = adv_inputs.clone()

        if is_malicious.any():
            X_mal = adv_inputs[is_malicious]
            y_mal = adv_labels[is_malicious].long()
            
            epsilon = random.uniform(0.01, 0.1)
            _, adv_mal, _ = attack(fmodel, X_mal, y_mal, epsilons=epsilon)

            adv_mal = X_mal + ((adv_mal - X_mal) * perturbation_mask)
            adv_mal = torch.clamp(adv_mal, x_min, x_max)

            # MAPE constraint 
            nonzero_mask = X_mal.abs() > 1e-2  
            per_feature_mape = torch.abs((adv_mal - X_mal) / (X_mal.abs() + 1e-8)) * 100.0
            mape_ok = ((per_feature_mape <= mape_threshold) | ~nonzero_mask).all(dim=1)
            adv_mal[~mape_ok] = X_mal[~mape_ok]

            adv_inputs_perturbed[is_malicious] = adv_mal

        # Combine clean batch + adversarial batch
        mixed_inputs = torch.cat([clean_inputs, adv_inputs_perturbed], dim=0)
        mixed_labels = torch.cat([clean_labels, adv_labels], dim=0)

        # OUTER MIN — update model weights
        model.train()
        optimizer.zero_grad()

        outputs = model(mixed_inputs).squeeze(-1)
        loss = criterion(outputs, mixed_labels.float())

        l1_reg = torch.stack([p.abs().sum() for p in model.parameters()]).sum()
        loss = loss + l1_lambda * l1_reg

        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        pbar.set_postfix({'loss': f"{loss.item():.4f}"})

    avg_train_loss = running_loss / min(len(train_clean_loader), len(train_adv_loader))

    # VALIDATION
    model.eval()
    val_running_loss = 0.0
    all_preds = []
    all_labels_list = []

    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs).squeeze(-1)
            v_loss = criterion(outputs, labels.float())
            val_running_loss += v_loss.item()
            v_probs = torch.sigmoid(outputs)
            v_predicted = (v_probs > 0.5).long()
            all_preds.extend(v_predicted.cpu().numpy())
            all_labels_list.extend(labels.cpu().numpy())

    avg_val_loss = val_running_loss / len(val_loader)
    val_accuracy = accuracy_score(all_labels_list, all_preds) * 100
    val_f1 = f1_score(all_labels_list, all_preds)

    print(f"Epoch {epoch+1}: Train Loss {avg_train_loss:.4f}, Val Loss {avg_val_loss:.4f}, Val Acc {val_accuracy:.2f}%, Val F1 {val_f1:.4f}")

# Save teacher model
torch.save(model.state_dict(), 'pgd_teacher_model.pth')
print("PGD teacher model saved")

# FINAL TEST
model.eval()
all_preds = []
all_labels_list = []

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs).squeeze(-1)
        probs = torch.sigmoid(outputs)
        predictions = (probs > 0.5).long()
        all_preds.extend(predictions.cpu().numpy())
        all_labels_list.extend(labels.cpu().numpy())

test_accuracy = accuracy_score(all_labels_list, all_preds) * 100
test_precision = precision_score(all_labels_list, all_preds)
test_recall = recall_score(all_labels_list, all_preds)
test_f1 = f1_score(all_labels_list, all_preds)
report = classification_report(all_labels_list, all_preds, target_names=['Attack', 'Benign'])

print("\n--- PGD TEACHER TEST METRICS ---")
print(f"Accuracy:  {test_accuracy:.2f}%")
print(f"Precision: {test_precision:.4f}")
print(f"Recall:    {test_recall:.4f}") 
print(f"F1 Score:  {test_f1:.4f}")
print("\nClassification Report:")
print(report)

/home/zawofeso/miniforge3/envs/IDS/lib/python3.10/site-packages/foolbox/models/pytorch.py:36: UserWarning: The PyTorch model is in training mode and therefore might not be deterministic. Call the eval() method to set it in evaluation mode if this is not intended.
  warnings.warn(
Epoch 1: 100%|██████████| 5373/5373 [08:21<00:00, 10.71it/s, loss=0.0741]


Epoch 1: Train Loss 0.0773, Val Loss 0.0771, Val Acc 98.04%, Val F1 0.9864


Epoch 2: 100%|██████████| 5373/5373 [08:22<00:00, 10.70it/s, loss=0.0827]


Epoch 2: Train Loss 0.0751, Val Loss 0.0782, Val Acc 98.05%, Val F1 0.9865


Epoch 3: 100%|██████████| 5373/5373 [08:24<00:00, 10.65it/s, loss=0.0707]


...

Epoch 8: Train Loss 0.0721, Val Loss 0.0743, Val Acc 98.03%, Val F1 0.9863


Epoch 9: 100%|██████████| 5373/5373 [08:18<00:00, 10.79it/s, loss=0.0854]


Epoch 9: Train Loss 0.0719, Val Loss 0.0781, Val Acc 98.07%, Val F1 0.9867


Epoch 10: 100%|██████████| 5373/5373 [08:23<00:00, 10.67it/s, loss=0.0968]


Epoch 10: Train Loss 0.0719, Val Loss 0.0746, Val Acc 98.04%, Val F1 0.9864
PGD teacher model saved

--- PGD TEACHER TEST METRICS ---
Accuracy:  98.03%
Precision: 0.9756
Recall:    0.9973
F1 Score:  0.9863

Classification Report:
              precision    recall  f1-score   support

      Attack       0.99      0.94      0.96    274823
      Benign       0.98      1.00      0.99    687691

    accuracy                           0.98    962514
   macro avg       0.98      0.97      0.98    962514
weighted avg       0.98      0.98      0.98    962514



#### Evaluation

In [27]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Get baseline model  
student_model = DDoSDetector(num_classes=1).to(device) 
student_model.load_state_dict(torch.load("mtkd_adr_student.pth", map_location=device))

<All keys matched successfully>

In [30]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Get baseline model  
union_model = DDoSDetector(num_classes=1).to(device) 
union_model.load_state_dict(torch.load("union_adv_model.pth", map_location=device))

<All keys matched successfully>

In [28]:
inputs, labels = next(iter(test_loader))
inputs = inputs.to(device)

wrapped_student = DDoSDetectorBinaryWrapper(student_model).to(device)
with torch.enable_grad():
    inputs_req = inputs[:4].clone().requires_grad_(True)
    out = wrapped_student(inputs_req)
    out.sum().backward()
    print("Student gradient norm:", inputs_req.grad.norm().item())

Student gradient norm: 224.24473571777344


In [17]:
def evaluate_under_attack(model, test_loader, attack, epsilon, device):
    model.eval()
    all_preds = []
    all_labels = []

    wrapped = DDoSDetectorBinaryWrapper(model).to(device)
    wrapped.eval()
    fmodel = fb.PyTorchModel(wrapped, bounds=(X_test.min().item(), X_test.max().item()))

    for inputs, labels in tqdm(test_loader, desc=f"Evaluating under attack ε={epsilon}"):
        inputs, labels = inputs.to(device), labels.to(device)

        is_malicious = (labels == 0)
        adv_inputs = inputs.clone()

        if is_malicious.sum() > 0:
            X_mal = inputs[is_malicious]
            y_mal = labels[is_malicious]
            
            _, adv_mal, _ = attack(fmodel, X_mal, y_mal, epsilons=epsilon)
            adv_mal = torch.clamp(adv_mal, X_test.min().item(), X_test.max().item())

            adv_inputs = inputs.clone()
            adv_inputs[is_malicious] = adv_mal
    
        with torch.no_grad():
            outputs = model(adv_inputs).squeeze(-1)
            probs = torch.sigmoid(outputs)
            preds = (probs > 0.5).long()

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    accuracy = accuracy_score(all_labels, all_preds) * 100
    f1 = f1_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds)
    recall = recall_score(all_labels, all_preds)
    report = classification_report(all_labels, all_preds, target_names=['Attack', 'Benign'])
    return accuracy, f1, precision, recall, report

In [31]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
#attack = LinfBasicIterativeAttack(abs_stepsize=0.01, steps=10)
#attack = PGD(rel_stepsize=0.02, steps=10, random_start=True)

attack = FGSM()
for epsilon in [0.01, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6]:
    accuracy, f1, precision, recall, report = evaluate_under_attack(union_model, test_loader, attack, epsilon, device)
    print(f"\n ε={epsilon}")
    print(f"Accuracy:  {accuracy:.2f}%")
    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")
    print(f"F1 Score:  {f1:.4f}")
    print(report)

Evaluating under attack ε=0.01: 100%|██████████| 470/470 [00:15<00:00, 30.05it/s]



 ε=0.01
Accuracy:  99.73%
Precision: 0.9980
Recall:    0.9982
F1 Score:  0.9981
              precision    recall  f1-score   support

      Attack       1.00      1.00      1.00    274823
      Benign       1.00      1.00      1.00    687691

    accuracy                           1.00    962514
   macro avg       1.00      1.00      1.00    962514
weighted avg       1.00      1.00      1.00    962514



Evaluating under attack ε=0.1: 100%|██████████| 470/470 [00:15<00:00, 30.87it/s]



 ε=0.1
Accuracy:  99.85%
Precision: 0.9997
Recall:    0.9982
F1 Score:  0.9989
              precision    recall  f1-score   support

      Attack       1.00      1.00      1.00    274823
      Benign       1.00      1.00      1.00    687691

    accuracy                           1.00    962514
   macro avg       1.00      1.00      1.00    962514
weighted avg       1.00      1.00      1.00    962514



Evaluating under attack ε=0.2: 100%|██████████| 470/470 [00:15<00:00, 30.24it/s]


...


 ε=0.4
Accuracy:  97.52%
Precision: 0.9681
Recall:    0.9982
F1 Score:  0.9829
              precision    recall  f1-score   support

      Attack       1.00      0.92      0.95    274823
      Benign       0.97      1.00      0.98    687691

    accuracy                           0.98    962514
   macro avg       0.98      0.96      0.97    962514
weighted avg       0.98      0.98      0.97    962514



Evaluating under attack ε=0.5: 100%|██████████| 470/470 [00:15<00:00, 31.23it/s]



 ε=0.5
Accuracy:  92.65%
Precision: 0.9080
Recall:    0.9982
F1 Score:  0.9510
              precision    recall  f1-score   support

      Attack       0.99      0.75      0.85    274823
      Benign       0.91      1.00      0.95    687691

    accuracy                           0.93    962514
   macro avg       0.95      0.87      0.90    962514
weighted avg       0.93      0.93      0.92    962514



Evaluating under attack ε=0.6: 100%|██████████| 470/470 [00:14<00:00, 31.74it/s]



 ε=0.6
Accuracy:  86.23%
Precision: 0.8394
Recall:    0.9982
F1 Score:  0.9119
              precision    recall  f1-score   support

      Attack       0.99      0.52      0.68    274823
      Benign       0.84      1.00      0.91    687691

    accuracy                           0.86    962514
   macro avg       0.92      0.76      0.80    962514
weighted avg       0.88      0.86      0.85    962514



In [32]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
#attack = LinfBasicIterativeAttack(abs_stepsize=0.01, steps=10)
#attack = PGD(rel_stepsize=0.02, steps=10, random_start=True)

attack = FGSM()
for epsilon in [0.01, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6]:
    accuracy, f1, precision, recall, report = evaluate_under_attack(student_model, test_loader, attack, epsilon, device)
    print(f"\n ε={epsilon}")
    print(f"Accuracy:  {accuracy:.2f}%")
    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")
    print(f"F1 Score:  {f1:.4f}")
    print(report)

Evaluating under attack ε=0.01: 100%|██████████| 470/470 [00:14<00:00, 31.39it/s]



 ε=0.01
Accuracy:  97.48%
Precision: 0.9682
Recall:    0.9975
F1 Score:  0.9826
              precision    recall  f1-score   support

      Attack       0.99      0.92      0.95    274823
      Benign       0.97      1.00      0.98    687691

    accuracy                           0.97    962514
   macro avg       0.98      0.96      0.97    962514
weighted avg       0.98      0.97      0.97    962514



Evaluating under attack ε=0.1: 100%|██████████| 470/470 [00:14<00:00, 32.98it/s]



 ε=0.1
Accuracy:  96.08%
Precision: 0.9502
Recall:    0.9975
F1 Score:  0.9733
              precision    recall  f1-score   support

      Attack       0.99      0.87      0.93    274823
      Benign       0.95      1.00      0.97    687691

    accuracy                           0.96    962514
   macro avg       0.97      0.93      0.95    962514
weighted avg       0.96      0.96      0.96    962514



Evaluating under attack ε=0.2: 100%|██████████| 470/470 [00:14<00:00, 31.62it/s]


...


 ε=0.4
Accuracy:  71.72%
Precision: 0.7172
Recall:    0.9975
F1 Score:  0.8344
              precision    recall  f1-score   support

      Attack       0.72      0.02      0.03    274823
      Benign       0.72      1.00      0.83    687691

    accuracy                           0.72    962514
   macro avg       0.72      0.51      0.43    962514
weighted avg       0.72      0.72      0.61    962514



Evaluating under attack ε=0.5: 100%|██████████| 470/470 [00:15<00:00, 29.70it/s]



 ε=0.5
Accuracy:  72.09%
Precision: 0.7199
Recall:    0.9975
F1 Score:  0.8362
              precision    recall  f1-score   support

      Attack       0.82      0.03      0.06    274823
      Benign       0.72      1.00      0.84    687691

    accuracy                           0.72    962514
   macro avg       0.77      0.51      0.45    962514
weighted avg       0.75      0.72      0.61    962514



Evaluating under attack ε=0.6: 100%|██████████| 470/470 [00:14<00:00, 32.52it/s]



 ε=0.6
Accuracy:  72.36%
Precision: 0.7219
Recall:    0.9975
F1 Score:  0.8376
              precision    recall  f1-score   support

      Attack       0.86      0.04      0.07    274823
      Benign       0.72      1.00      0.84    687691

    accuracy                           0.72    962514
   macro avg       0.79      0.52      0.46    962514
weighted avg       0.76      0.72      0.62    962514



## Multi-Teacher Knowledge Distillation

In [19]:
class DDoSDetector(nn.Module):
    def __init__(self, num_classes=1):
        super().__init__()
        layers = []
        in_channels = 1
        for _ in range(10):
            layers.append(nn.Conv1d(in_channels, 108, kernel_size=5, padding=2))
            layers.append(nn.BatchNorm1d(108))
            layers.append(nn.ReLU())
            in_channels = 108
        self.cnn_backbone = nn.Sequential(*layers)
        self.global_pool = nn.AdaptiveAvgPool1d(1)
        self.classifier = nn.Linear(108, num_classes)

    def forward(self, x, return_features=False):
        x = x.unsqueeze(1)
        x = self.cnn_backbone(x)
        features = self.global_pool(x).squeeze(-1)  # (B, 108)
        logit = self.classifier(features)            # (B, 1)
        if return_features:
            return logit, features
        return logit

In [20]:
def compute_adaptive_weights(student_logits, teacher_logits_list):
    similarities = []
    for t_logits in teacher_logits_list:
        sim = F.cosine_similarity(student_logits, t_logits, dim=0).mean()
        similarities.append(sim)

    weights = [1.0 + s for s in similarities]
    total = sum(weights)
    normalized = [w / total for w in weights]
    return normalized


def distillation_loss(student_logits, teacher_logits_list, weights, temperature=4.0):
    s_soft = torch.sigmoid(student_logits / temperature)
    student_probs = torch.cat([1 - s_soft, s_soft], dim=1)
    student_log_probs = torch.log(student_probs.clamp(min=1e-8))

    weighted_t_soft = torch.zeros_like(s_soft)
    for t_logits, w in zip(teacher_logits_list, weights):
        t_soft = torch.sigmoid(t_logits / temperature)
        weighted_t_soft += w * t_soft
    teacher_probs = torch.cat([1 - weighted_t_soft, weighted_t_soft], dim=1)

    kld = F.kl_div(student_log_probs, teacher_probs.clamp(min=1e-8), reduction='batchmean')
    return kld * (temperature ** 2)


def feature_distillation_loss(student_features, teacher_features_list, weights):
    loss = torch.tensor(0.0, device=student_features.device)
    for t_features, w in zip(teacher_features_list, weights):
        loss += w * F.mse_loss(student_features, t_features.detach())
    return loss

In [22]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
fgsm_teacher = DDoSDetector(num_classes=1).to(device)
fgsm_teacher.load_state_dict(torch.load('fgsm_teacher_model.pth', map_location=device))
fgsm_teacher.eval()

ifgsm_teacher = DDoSDetector(num_classes=1).to(device)
ifgsm_teacher.load_state_dict(torch.load('ifgsm_teacher_model.pth', map_location=device))
ifgsm_teacher.eval()

pgd_teacher = DDoSDetector(num_classes=1).to(device)
pgd_teacher.load_state_dict(torch.load('pgd_teacher_model.pth', map_location=device))
pgd_teacher.eval()

teachers = [fgsm_teacher, ifgsm_teacher, pgd_teacher]

In [ ]:
import random

fgsm_attack  = FGSM()
ifgsm_attack = LinfBasicIterativeAttack(abs_stepsize=0.01, steps=20)
pgd_attack   = PGD(rel_stepsize=0.02, steps=40, random_start=True)
attack_list  = [fgsm_attack, ifgsm_attack, pgd_attack]
epsilon_range = [0.03, 0.07, 0.1]

# Precompute foolbox models for each teacher
fmodels = []
for teacher in teachers:
    wrapped = DDoSDetectorBinaryWrapper(teacher).to(device)
    wrapped.eval()
    fmodel = fb.PyTorchModel(wrapped, bounds=(0.0, 1.0))
    fmodels.append(fmodel)

student = DDoSDetector(num_classes=1).to(device)
criterion_ce = nn.BCEWithLogitsLoss()
optimizer    = optim.Adam(student.parameters(), lr=1e-3, weight_decay=1e-5)

alpha  = 0.4   
beta   = 0.4  
epochs = 6

train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=1024, shuffle=True)
val_loader   = DataLoader(TensorDataset(X_val,   y_val),   batch_size=1024, shuffle=False)
test_loader  = DataLoader(TensorDataset(X_test,  y_test),  batch_size=1024, shuffle=False)

for epoch in range(epochs):
    student.train()
    for t in teachers:
        t.eval()

    running_loss = 0.0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}")

    for inputs, labels in pbar:
        inputs = inputs.to(device)
        labels = labels.to(device)

        is_malicious = (labels == 0)
        epsilon = random.choice(epsilon_range)

        # Generate adversarial batch per attack/teacher pair
        adv_batches = []
        for attack, fmodel_t in zip(attack_list, fmodels):
            adv_batch = inputs.clone()
            if is_malicious.sum() > 0:
                X_mal = inputs[is_malicious]
                y_mal = labels[is_malicious]
                with torch.enable_grad():
                    _, adv_mal, _ = attack(fmodel_t, X_mal, y_mal.long(), epsilons=epsilon)
                adv_mal = torch.clamp(adv_mal, 0.0, 1.0)
                adv_batch = inputs.clone()
                adv_batch[is_malicious] = adv_mal
            adv_batches.append(adv_batch)

        # Stack: clean + fgsm_adv + ifgsm_adv + pgd_adv
        all_inputs = torch.cat([inputs] + adv_batches, dim=0)   # (4B, features)
        all_labels = torch.cat([labels] * 4, dim=0)             # (4B,)

        optimizer.zero_grad()

        # Student forward — get both logits and features
        student_logits, student_features = student(all_inputs, return_features=True)

        # Teacher forwards
        with torch.no_grad():
            teacher_logits_list   = []
            teacher_features_list = []
            for t in teachers:
                t_logits, t_features = t(all_inputs, return_features=True)
                teacher_logits_list.append(t_logits)
                teacher_features_list.append(t_features)

        # Adaptive weights from logits
        weights = compute_adaptive_weights(student_logits.detach(), teacher_logits_list)

        # Losses
        ce_loss      = criterion_ce(student_logits.squeeze(-1), all_labels.float())
        kd_loss      = distillation_loss(student_logits, teacher_logits_list, weights)
        feat_loss    = feature_distillation_loss(student_features, teacher_features_list, weights)
        total_loss   = (1 - alpha - beta) * ce_loss + alpha * kd_loss + beta * feat_loss

        total_loss.backward()
        optimizer.step()

        running_loss += total_loss.item()
        pbar.set_postfix({'loss':  f"{total_loss.item():.4f}",
                          'ce':    f"{ce_loss.item():.4f}",
                          'kd':    f"{kd_loss.item():.4f}",
                          'feat':  f"{feat_loss.item():.4f}",
                          'eps':   f"{epsilon:.2f}"})

    avg_loss = running_loss / len(train_loader)

    # Diagnostics
    student.eval()
    diag_inputs, _ = next(iter(train_loader))
    diag_inputs = diag_inputs.to(device)

    with torch.no_grad():
        s_logits, s_features = student(diag_inputs, return_features=True)
        t_logits_list, t_features_list = [], []
        for t in teachers:
            tl, tf = t(diag_inputs, return_features=True)
            t_logits_list.append(tl)
            t_features_list.append(tf)

        T = 4.0
        s_soft = torch.sigmoid(s_logits / T)
        student_probs = torch.cat([1 - s_soft, s_soft], dim=1)
        student_log_probs = torch.log(student_probs.clamp(min=1e-8))

        w = compute_adaptive_weights(s_logits, t_logits_list)
        weighted_t_soft = torch.zeros_like(s_soft)
        for tl, wi in zip(t_logits_list, w):
            weighted_t_soft += wi * torch.sigmoid(tl / T)
        teacher_probs = torch.cat([1 - weighted_t_soft, weighted_t_soft], dim=1)
        kld = F.kl_div(student_log_probs, teacher_probs.clamp(min=1e-8), reduction='batchmean')

        # Feature similarity between student and each teacher
        feat_sims = []
        for tf in t_features_list:
            sim = F.cosine_similarity(s_features, tf, dim=1).mean().item()
            feat_sims.append(round(sim, 4))

        print(f"\n── Epoch {epoch+1} Diagnostics ──")
        print(f"Student logits:  {s_logits.min().item():.2f} to {s_logits.max().item():.2f}")
        print(f"Student soft:    {s_soft.min().item():.3f} to {s_soft.max().item():.3f}")
        for i, tl in enumerate(t_logits_list):
            t_soft = torch.sigmoid(tl / T)
            print(f"Teacher {i} logits: {tl.min().item():.2f} to {tl.max().item():.2f} | soft: {t_soft.min().item():.3f} to {t_soft.max().item():.3f}")
        print(f"Adaptive weights: {[round(wi.item(), 3) for wi in w]}")
        print(f"KLD (student→weighted teacher): {kld.item():.4f}")
        for i, tl in enumerate(t_logits_list):
            t_soft_i = torch.sigmoid(tl / T)
            t_probs_i = torch.cat([1 - t_soft_i, t_soft_i], dim=1)
            kld_i = F.kl_div(student_log_probs, t_probs_i.clamp(min=1e-8), reduction='batchmean')
            print(f"KLD (student→teacher {i}): {kld_i.item():.4f}")
        for i in range(len(t_logits_list)):
            for j in range(i+1, len(t_logits_list)):
                ti_soft = torch.sigmoid(t_logits_list[i] / T)
                tj_soft = torch.sigmoid(t_logits_list[j] / T)
                ti_probs = torch.cat([1 - ti_soft, ti_soft], dim=1)
                tj_probs = torch.cat([1 - tj_soft, tj_soft], dim=1)
                ti_log = torch.log(ti_probs.clamp(min=1e-8))
                kld_ij = F.kl_div(ti_log, tj_probs.clamp(min=1e-8), reduction='batchmean')
                print(f"KLD (teacher {i}→teacher {j}): {kld_ij.item():.4f}")
        t_preds = [(torch.sigmoid(tl) > 0.5).long() for tl in t_logits_list]
        agree_all = (t_preds[0] == t_preds[1]) & (t_preds[1] == t_preds[2])
        print(f"Teacher agreement (all 3): {agree_all.float().mean().item()*100:.1f}%")
        majority = (t_preds[0].float() + t_preds[1].float() + t_preds[2].float()) >= 2
        s_preds = (torch.sigmoid(s_logits) > 0.5).long()
        student_majority_agree = (s_preds.squeeze() == majority.squeeze()).float().mean().item()
        print(f"Student vs majority vote agreement: {student_majority_agree*100:.1f}%")
        print(f"Weighted teacher soft mean: {weighted_t_soft.mean().item():.3f}")
        print(f"Weighted teacher soft std:  {weighted_t_soft.std().item():.3f}")
        print(f"Feature cosine sim (student→teachers): {feat_sims}")

    # Validation
    all_preds, all_labels_list = [], []
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            logits = student(inputs).squeeze(-1)
            preds  = (torch.sigmoid(logits) > 0.5).long()
            all_preds.extend(preds.cpu().numpy())
            all_labels_list.extend(labels.cpu().numpy())

    val_acc = accuracy_score(all_labels_list, all_preds) * 100
    val_f1  = f1_score(all_labels_list, all_preds)
    print(f"Epoch {epoch+1}: Loss {avg_loss:.4f} | Val Acc {val_acc:.2f}% | Val F1 {val_f1:.4f}")

torch.save(student.state_dict(), 'mtkd_adr_student.pth')
print("Student model saved.")

# Final test
student.eval()
all_preds, all_labels_list = [], []
with torch.no_grad():
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        logits = student(inputs).squeeze(-1)
        preds  = (torch.sigmoid(logits) > 0.5).long()
        all_preds.extend(preds.cpu().numpy())
        all_labels_list.extend(labels.cpu().numpy())

print("\n--- MTKD-ADR STUDENT TEST METRICS (Clean) ---")
print(classification_report(all_labels_list, all_preds, target_names=['Attack', 'Benign']))

Epoch 1/6:   0%|          | 0/10746 [00:00<?, ?it/s]

Epoch 1/6: 100%|██████████| 10746/10746 [1:20:23<00:00,  2.23it/s, loss=0.0472, ce=0.1324, kd=0.0444, feat=0.0073, eps=0.03]



── Epoch 1 Diagnostics ──
Student logits:  -12.32 to 10.40
Student soft:    0.044 to 0.931
Teacher 0 logits: -14.44 to 11.07 | soft: 0.026 to 0.941
Teacher 1 logits: -13.47 to 11.21 | soft: 0.033 to 0.943
Teacher 2 logits: -17.74 to 10.67 | soft: 0.012 to 0.935
Adaptive weights: [0.334, 0.334, 0.332]
KLD (student→weighted teacher): 0.0009
KLD (student→teacher 0): 0.0016
KLD (student→teacher 1): 0.0024
KLD (student→teacher 2): 0.0060
KLD (teacher 0→teacher 1): 0.0053
KLD (teacher 0→teacher 2): 0.0096
KLD (teacher 1→teacher 2): 0.0052
Teacher agreement (all 3): 99.7%
Student vs majority vote agreement: 99.9%
Weighted teacher soft mean: 0.451
Weighted teacher soft std:  0.326
Feature cosine sim (student→teachers): [0.9792, 0.9872, 0.9792]
Epoch 1: Loss 0.0630 | Val Acc 98.02% | Val F1 0.9863


Epoch 2/6: 100%|██████████| 10746/10746 [1:18:17<00:00,  2.29it/s, loss=0.0376, ce=0.1050, kd=0.0336, feat=0.0079, eps=0.03]



── Epoch 2 Diagnostics ──
Student logits:  -13.25 to 10.78
Student soft:    0.035 to 0.937
Teacher 0 logits: -13.19 to 11.23 | soft: 0.036 to 0.943
Teacher 1 logits: -14.08 to 11.43 | soft: 0.029 to 0.946
Teacher 2 logits: -17.62 to 10.67 | soft: 0.012 to 0.935
Adaptive weights: [0.333, 0.334, 0.333]
KLD (student→weighted teacher): 0.0001
KLD (student→teacher 0): 0.0030
KLD (student→teacher 1): 0.0010
KLD (student→teacher 2): 0.0030
KLD (teacher 0→teacher 1): 0.0049
KLD (teacher 0→teacher 2): 0.0092
KLD (teacher 1→teacher 2): 0.0040
Teacher agreement (all 3): 100.0%
Student vs majority vote agreement: 100.0%
Weighted teacher soft mean: 0.471
Weighted teacher soft std:  0.321
Feature cosine sim (student→teachers): [0.9826, 0.9914, 0.9849]
Epoch 2: Loss 0.0411 | Val Acc 98.02% | Val F1 0.9863


Epoch 3/6:  10%|▉         | 1061/10746 [07:43<1:10:45,  2.28it/s, loss=0.0496, ce=0.1694, kd=0.0337, feat=0.0056, eps=0.10]

In [29]:
# Load hardened teacher models
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

fgsm_teacher = DDoSDetector(num_classes=1).to(device)
fgsm_teacher.load_state_dict(torch.load('fgsm_teacher_model.pth', map_location=device))
fgsm_teacher.eval()

ifgsm_teacher = DDoSDetector(num_classes=1).to(device)
ifgsm_teacher.load_state_dict(torch.load('ifgsm_teacher_model.pth', map_location=device))
ifgsm_teacher.eval()

pgd_teacher = DDoSDetector(num_classes=1).to(device)
pgd_teacher.load_state_dict(torch.load('pgd_teacher_model.pth', map_location=device))
pgd_teacher.eval()

teachers = [fgsm_teacher, ifgsm_teacher, pgd_teacher]

# Adaptive weight computation (cosine similarity)
def compute_adaptive_weights(student_logits, teacher_logits_list):
    similarities = []
    for t_logits in teacher_logits_list:
        # cosine similarity per sample, then mean over batch
        sim = F.cosine_similarity(student_logits, t_logits, dim=0).mean()
        similarities.append(sim)

    # w_i = 1 + similarity (Eq. 12 in paper)
    weights = [1.0 + s for s in similarities]

    # normalize so weights sum to 1 (Eq. 13)
    total = sum(weights)
    normalized = [w / total for w in weights]
    return normalized

# Knowledge Distillation Loss (KL Divergence)
def distillation_loss(student_logits, teacher_logits_list, weights, temperature=16.0):
    # Convert to 2-class distributions (B, 2)
    s_soft = torch.sigmoid(student_logits / temperature)
    student_probs = torch.cat([1 - s_soft, s_soft], dim=1)             # (B, 2)
    student_log_probs = torch.log(student_probs.clamp(min=1e-8))

    # Weighted teacher targets (B, 2)
    weighted_t_soft = torch.zeros_like(s_soft)
    for t_logits, w in zip(teacher_logits_list, weights):
        t_soft = torch.sigmoid(t_logits / temperature)
        weighted_t_soft += w * t_soft
    teacher_probs = torch.cat([1 - weighted_t_soft, weighted_t_soft], dim=1)

    kld = F.kl_div(student_log_probs, teacher_probs.clamp(min=1e-8), reduction='batchmean')
    return kld * (temperature ** 2)

# Student model + training setup
student = DDoSDetector(num_classes=1).to(device)

criterion_ce  = nn.BCEWithLogitsLoss()
optimizer     = optim.Adam(student.parameters(), lr=1e-3, weight_decay=1e-5)

alpha  = 0.8      
epochs = 10
kd_start_epoch = 3  

# Clean data only for distillation (as per paper)
train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=1024, shuffle=True)
val_loader   = DataLoader(TensorDataset(X_val,   y_val),   batch_size=1024, shuffle=False)
test_loader  = DataLoader(TensorDataset(X_test,  y_test),  batch_size=1024, shuffle=False)

# Training loop
for epoch in range(epochs):
    student.train()
    for t in teachers:
        t.eval()

    running_loss = 0.0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}")

    for inputs, labels in pbar:
        inputs = inputs.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        # Student forward
        student_logits = student(inputs)                                # (B, 1)

        # Teacher forwards (no grad needed)
        with torch.no_grad():
            teacher_logits_list = [t(inputs) for t in teachers]        # list of (B, 1)
        
        ce_loss = criterion_ce(student_logits.squeeze(-1), labels.float())
        kd_loss = torch.tensor(0.0, device=device)

        if epoch < kd_start_epoch:
            total_loss = ce_loss
        else:
            weights = compute_adaptive_weights(student_logits.detach(), teacher_logits_list)
            kd_loss   = distillation_loss(student_logits, teacher_logits_list, weights, temperature=16.0)
            total_loss = (1 - alpha) * ce_loss + alpha * kd_loss

        total_loss.backward()
        optimizer.step()

        running_loss += total_loss.item()
        pbar.set_postfix({'loss': f"{total_loss.item():.4f}",
                          'ce':   f"{ce_loss.item():.4f}",
                          'kd':   f"{kd_loss.item():.4f}"})

    avg_loss = running_loss / len(train_loader)

    # Validation
    student.eval()
    diag_inputs, diag_labels = next(iter(train_loader))
    diag_inputs = diag_inputs.to(device)

    # diagnostic in no_grad
    with torch.no_grad():
        s_logits = student(diag_inputs)
        t_logits_list = [t(diag_inputs) for t in teachers]
        
        T = 16.0
        s_soft = torch.sigmoid(s_logits / T)
        student_probs = torch.cat([1 - s_soft, s_soft], dim=1)
        student_log_probs = torch.log(student_probs.clamp(min=1e-8))
        
        w = compute_adaptive_weights(s_logits, t_logits_list)
        weighted_t_soft = torch.zeros_like(s_soft)
        for tl, wi in zip(t_logits_list, w):
            weighted_t_soft += wi * torch.sigmoid(tl / T)
        teacher_probs = torch.cat([1 - weighted_t_soft, weighted_t_soft], dim=1)
        kld = F.kl_div(student_log_probs, teacher_probs.clamp(min=1e-8), reduction='batchmean')
        print(f"\n── Epoch {epoch+1} Diagnostics ──")
        
        # Student
        print(f"Student logits:  {s_logits.min().item():.2f} to {s_logits.max().item():.2f}")
        print(f"Student soft:    {s_soft.min().item():.3f} to {s_soft.max().item():.3f}")
        
        # Teachers
        for i, tl in enumerate(t_logits_list):
            t_soft = torch.sigmoid(tl / T)
            print(f"Teacher {i} logits: {tl.min().item():.2f} to {tl.max().item():.2f} | soft: {t_soft.min().item():.3f} to {t_soft.max().item():.3f}")

        # Adaptive weights
        print(f"Adaptive weights: {[round(wi.item(), 3) for wi in w]}")

        # KLD between student and weighted teacher
        print(f"KLD (student→teacher): {kld.item():.4f}")

        # KLD between each individual teacher and student
        for i, tl in enumerate(t_logits_list):
            t_soft_i = torch.sigmoid(tl / T)
            t_probs_i = torch.cat([1 - t_soft_i, t_soft_i], dim=1)
            kld_i = F.kl_div(student_log_probs, t_probs_i.clamp(min=1e-8), reduction='batchmean')
            print(f"KLD (student→teacher {i}): {kld_i.item():.4f}")

        # Agreement between teachers — are they consistent with each other?
        for i in range(len(t_logits_list)):
            for j in range(i+1, len(t_logits_list)):
                ti_soft = torch.sigmoid(t_logits_list[i] / T)
                tj_soft = torch.sigmoid(t_logits_list[j] / T)
                ti_probs = torch.cat([1 - ti_soft, ti_soft], dim=1)
                tj_probs = torch.cat([1 - tj_soft, tj_soft], dim=1)
                ti_log = torch.log(ti_probs.clamp(min=1e-8))
                kld_ij = F.kl_div(ti_log, tj_probs.clamp(min=1e-8), reduction='batchmean')
                print(f"KLD (teacher {i}→teacher {j}): {kld_ij.item():.4f}")

        # Teacher prediction agreement on this batch — what % do all 3 agree on?
        t_preds = [(torch.sigmoid(tl) > 0.5).long() for tl in t_logits_list]
        agree_all = (t_preds[0] == t_preds[1]) & (t_preds[1] == t_preds[2])
        print(f"Teacher agreement (all 3): {agree_all.float().mean().item()*100:.1f}%")
        
        # Student vs teacher majority vote agreement
        majority = (t_preds[0].float() + t_preds[1].float() + t_preds[2].float()) >= 2
        s_preds = (torch.sigmoid(s_logits) > 0.5).long()
        student_majority_agree = (s_preds.squeeze() == majority.squeeze()).float().mean().item()
        print(f"Student vs majority vote agreement: {student_majority_agree*100:.1f}%")
        
        # Weighted teacher soft distribution stats — is it well spread or collapsed?
        print(f"Weighted teacher soft mean: {weighted_t_soft.mean().item():.3f}")
        print(f"Weighted teacher soft std:  {weighted_t_soft.std().item():.3f}")

    all_preds, all_labels_list = [], []
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            logits = student(inputs).squeeze(-1)
            preds  = (torch.sigmoid(logits) > 0.5).long()
            all_preds.extend(preds.cpu().numpy())
            all_labels_list.extend(labels.cpu().numpy())

    val_acc = accuracy_score(all_labels_list, all_preds) * 100
    val_f1  = f1_score(all_labels_list, all_preds)
    print(f"Epoch {epoch+1}: Loss {avg_loss:.4f} | Val Acc {val_acc:.2f}% | Val F1 {val_f1:.4f}")

torch.save(student.state_dict(), 'mtkd_adr_student.pth')
print("Student model saved.")

# Final test on clean data
student.eval()
all_preds, all_labels_list = [], []
with torch.no_grad():
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        logits = student(inputs).squeeze(-1)
        preds  = (torch.sigmoid(logits) > 0.5).long()
        all_preds.extend(preds.cpu().numpy())

        all_labels_list.extend(labels.cpu().numpy())

print("\n--- MTKD-ADR STUDENT TEST METRICS (Clean) ---")
print(classification_report(all_labels_list, all_preds, target_names=['Attack', 'Benign']))

Epoch 1/10:   0%|          | 0/10746 [00:00<?, ?it/s]

Epoch 1/10: 100%|██████████| 10746/10746 [04:25<00:00, 40.50it/s, loss=0.1152, ce=0.1152, kd=0.0000]



── Epoch 1 Diagnostics ──
Student logits:  -13.34 to 10.06
Student soft:    0.303 to 0.652
Teacher 0 logits: -12.54 to 10.25 | soft: 0.313 to 0.655
Teacher 1 logits: -12.41 to 11.09 | soft: 0.315 to 0.667
Teacher 2 logits: -14.23 to 11.02 | soft: 0.291 to 0.666
Adaptive weights: [0.333, 0.332, 0.334]
KLD (student→teacher): 0.0006
KLD (student→teacher 0): 0.0007
KLD (student→teacher 1): 0.0010
KLD (student→teacher 2): 0.0005
KLD (teacher 0→teacher 1): 0.0005
KLD (teacher 0→teacher 2): 0.0004
KLD (teacher 1→teacher 2): 0.0004
Teacher agreement (all 3): 99.1%
Student vs majority vote agreement: 98.9%
Weighted teacher soft mean: 0.481
Weighted teacher soft std:  0.107
Epoch 1: Loss 0.1071 | Val Acc 97.70% | Val F1 0.9840


Epoch 2/10: 100%|██████████| 10746/10746 [04:20<00:00, 41.32it/s, loss=0.1224, ce=0.1224, kd=0.0000]



── Epoch 2 Diagnostics ──
Student logits:  -11.90 to 10.11
Student soft:    0.322 to 0.653
Teacher 0 logits: -13.32 to 10.26 | soft: 0.303 to 0.655
Teacher 1 logits: -12.46 to 11.16 | soft: 0.315 to 0.668
Teacher 2 logits: -13.15 to 11.01 | soft: 0.305 to 0.666
Adaptive weights: [0.333, 0.333, 0.334]
KLD (student→teacher): 0.0004
KLD (student→teacher 0): 0.0005
KLD (student→teacher 1): 0.0007
KLD (student→teacher 2): 0.0005
KLD (teacher 0→teacher 1): 0.0005
KLD (teacher 0→teacher 2): 0.0004
KLD (teacher 1→teacher 2): 0.0004
Teacher agreement (all 3): 99.0%
Student vs majority vote agreement: 99.6%
Weighted teacher soft mean: 0.479
Weighted teacher soft std:  0.105
Epoch 2: Loss 0.1001 | Val Acc 98.07% | Val F1 0.9866


...


── Epoch 5 Diagnostics ──
Student logits:  -11.69 to 9.97
Student soft:    0.325 to 0.651
Teacher 0 logits: -13.14 to 10.40 | soft: 0.306 to 0.657
Teacher 1 logits: -12.42 to 10.98 | soft: 0.315 to 0.665
Teacher 2 logits: -11.89 to 11.04 | soft: 0.322 to 0.666
Adaptive weights: [0.333, 0.333, 0.333]
KLD (student→teacher): 0.0000
KLD (student→teacher 0): 0.0002
KLD (student→teacher 1): 0.0001
KLD (student→teacher 2): 0.0002
KLD (teacher 0→teacher 1): 0.0004
KLD (teacher 0→teacher 2): 0.0004
KLD (teacher 1→teacher 2): 0.0004
Teacher agreement (all 3): 99.3%
Student vs majority vote agreement: 99.9%
Weighted teacher soft mean: 0.483
Weighted teacher soft std:  0.107
Epoch 5: Loss 0.0233 | Val Acc 98.05% | Val F1 0.9865


Epoch 6/10: 100%|██████████| 10746/10746 [04:30<00:00, 39.79it/s, loss=0.0218, ce=0.1014, kd=0.0020]



── Epoch 6 Diagnostics ──
Student logits:  -13.32 to 10.34
Student soft:    0.303 to 0.656
Teacher 0 logits: -13.66 to 10.54 | soft: 0.299 to 0.659
Teacher 1 logits: -14.49 to 10.87 | soft: 0.288 to 0.664
Teacher 2 logits: -13.69 to 11.06 | soft: 0.298 to 0.666
Adaptive weights: [0.333, 0.333, 0.333]
KLD (student→teacher): 0.0000
KLD (student→teacher 0): 0.0001
KLD (student→teacher 1): 0.0001
KLD (student→teacher 2): 0.0001
KLD (teacher 0→teacher 1): 0.0004
KLD (teacher 0→teacher 2): 0.0004
KLD (teacher 1→teacher 2): 0.0004
Teacher agreement (all 3): 99.4%
Student vs majority vote agreement: 100.0%
Weighted teacher soft mean: 0.473
Weighted teacher soft std:  0.106
Epoch 6: Loss 0.0224 | Val Acc 98.08% | Val F1 0.9867


Epoch 7/10:   7%|▋         | 717/10746 [00:18<04:19, 38.72it/s, loss=0.0149, ce=0.0644, kd=0.0025]


KeyboardInterrupt: 

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Get baseline model  
student_model = DDoSDetector(num_classes=1).to(device) 
student_model.load_state_dict(torch.load("mtkd_adr_student.pth", map_location=device))

In [27]:
# features_to_peturb = ['Flow Duration', 'Flow IAT Mean', 'Flow IAT Max', 'Flow IAT Min', 'Flow IAT Std',
#                       'Fwd IAT Tot', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min',
#                       'Bwd IAT Tot', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min',
#                       'Active Mean', 'Active Std', 'Active Max', 'Active Min',
#                       'Idle Mean', 'Idle Std', 'Idle Max', 'Idle Min'   
# ]

# feature_indices = [numerical_cols.index(feature) for feature in features_to_peturb]

# attack_list = [(FGSM(), "FGSM"), (LinfBasicIterativeAttack(abs_stepsize=0.01, steps=20), "IFGSM"), (PGD(rel_stepsize=0.02, steps=40, random_start=True), "PGD")]


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
attack = FGSM()
for epsilon in [0.03, 0.07, 0.1]:
    accuracy, f1, precision, recall, report = evaluate_under_attack(student_model, test_loader, attack, epsilon, device)
    print(f"\n ε={epsilon}")
    print(f"Accuracy:  {accuracy:.2f}%")
    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")
    print(f"F1 Score:  {f1:.4f}")
    print(report)

Evaluating under attack ε=0.03:   0%|          | 2/940 [00:00<00:56, 16.52it/s]

Evaluating under attack ε=0.03: 100%|██████████| 940/940 [00:18<00:00, 50.53it/s]



 ε=0.03
Accuracy:  78.95%
Precision: 0.7734
Recall:    0.9977
F1 Score:  0.8713
              precision    recall  f1-score   support

      Attack       0.98      0.27      0.42    274823
      Benign       0.77      1.00      0.87    687691

    accuracy                           0.79    962514
   macro avg       0.88      0.63      0.65    962514
weighted avg       0.83      0.79      0.74    962514



Evaluating under attack ε=0.07: 100%|██████████| 940/940 [00:16<00:00, 55.65it/s]



 ε=0.07
Accuracy:  75.09%
Precision: 0.7423
Recall:    0.9977
F1 Score:  0.8513
              precision    recall  f1-score   support

      Attack       0.96      0.13      0.23    274823
      Benign       0.74      1.00      0.85    687691

    accuracy                           0.75    962514
   macro avg       0.85      0.57      0.54    962514
weighted avg       0.80      0.75      0.68    962514



Evaluating under attack ε=0.1:  20%|█▉        | 185/940 [00:02<00:10, 69.24it/s]

In [21]:
inputs, labels = next(iter(train_loader))
inputs, labels = inputs.to(device), labels.to(device)

student.eval()
with torch.no_grad():
    student_logits = student(inputs)
    teacher_logits_list = [t(inputs) for t in teachers]

print("Student logits range:", student_logits.min().item(), "to", student_logits.max().item())

for i, t_logits in enumerate(teacher_logits_list):
    print(f"Teacher {i} logits range:", t_logits.min().item(), "to", t_logits.max().item())

# Check soft probs
T = 4.0
student_soft = torch.sigmoid(student_logits / T)
print("\nStudent soft probs range:", student_soft.min().item(), "to", student_soft.max().item())

for i, t_logits in enumerate(teacher_logits_list):
    t_soft = torch.sigmoid(t_logits / T)
    print(f"Teacher {i} soft probs range:", t_soft.min().item(), "to", t_soft.max().item())

# Check weighted target
weights = compute_adaptive_weights(student_logits.detach(), teacher_logits_list)
print("\nWeights:", weights)

weighted_teacher_soft = torch.zeros_like(student_soft)
for t_logits, w in zip(teacher_logits_list, weights):
    t_soft = torch.sigmoid(t_logits / T)
    weighted_teacher_soft += w * t_soft

print("Weighted teacher soft range:", weighted_teacher_soft.min().item(), "to", weighted_teacher_soft.max().item())

# Check KL directly
student_log = torch.log(student_soft.clamp(min=1e-8))
print("Student log probs range:", student_log.min().item(), "to", student_log.max().item())

kld = F.kl_div(student_log, weighted_teacher_soft.clamp(min=1e-8), reduction='batchmean')
print("\nRaw KLD:", kld.item())

Student logits range: 0.5289610028266907 to 29.228302001953125
Teacher 0 logits range: -12.470650672912598 to 10.506449699401855
Teacher 1 logits range: -13.373165130615234 to 11.141000747680664
Teacher 2 logits range: -12.664570808410645 to 11.018831253051758

Student soft probs range: 0.5330119729042053 to 0.999329686164856
Teacher 0 soft probs range: 0.04238453879952431 to 0.9325547814369202
Teacher 1 soft probs range: 0.034115538001060486 to 0.9418734908103943
Teacher 2 soft probs range: 0.040459901094436646 to 0.9401786923408508

Weights: [tensor(0.3347, device='cuda:0'), tensor(0.3393, device='cuda:0'), tensor(0.3260, device='cuda:0')]
Weighted teacher soft range: 0.04497915506362915 to 0.9308385848999023
Student log probs range: -0.62921142578125 to -0.0006705385749228299

Raw KLD: -0.21721285581588745


In [41]:
# Load hardened teacher models
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

fgsm_teacher = DDoSDetector(num_classes=1).to(device)
fgsm_teacher.load_state_dict(torch.load('fgsm_teacher_model.pth', map_location=device))
fgsm_teacher.eval()

ifgsm_teacher = DDoSDetector(num_classes=1).to(device)
ifgsm_teacher.load_state_dict(torch.load('ifgsm_teacher_model.pth', map_location=device))
ifgsm_teacher.eval()

pgd_teacher = DDoSDetector(num_classes=1).to(device)
pgd_teacher.load_state_dict(torch.load('pgd_teacher_model.pth', map_location=device))
pgd_teacher.eval()

teachers = [fgsm_teacher, ifgsm_teacher, pgd_teacher]

def compute_adaptive_weights(student_logits, teacher_logits_list):
    weights = []
    for t_logits in teacher_logits_list:
        t_prob = torch.sigmoid(t_logits)
        # confidence = how far from 0.5 the teacher is
        confidence = torch.abs(t_prob - 0.5).mean()
        weights.append(confidence)
    
    total = sum(weights)
    normalized = [w / total for w in weights]
    return normalized

# Knowledge Distillation Loss (KL Divergence)
def distillation_loss(student_logits, teacher_logits_list, weights, temperature=4.0):
    # Convert to 2-class distributions [p_attack, p_benign]
    student_p = torch.sigmoid(student_logits / temperature)
    student_dist = torch.cat([1 - student_p, student_p], dim=1)  # (B, 2)
    
    weighted_teacher_dist = torch.zeros_like(student_dist)
    for t_logits, w in zip(teacher_logits_list, weights):
        t_p = torch.sigmoid(t_logits / temperature)
        t_dist = torch.cat([1 - t_p, t_p], dim=1)  # (B, 2)
        weighted_teacher_dist += w * t_dist

    kld = F.kl_div(
        torch.log(student_dist.clamp(min=1e-8)),
        weighted_teacher_dist,
        reduction='batchmean'
    )
    return kld * (temperature ** 2)

In [42]:
# Student model + training setup
student = DDoSDetector(num_classes=1).to(device)

criterion_ce  = nn.BCEWithLogitsLoss()
optimizer     = optim.Adam(student.parameters(), lr=1e-3, weight_decay=1e-5)

alpha  = 0.7      
epochs = 5

# Clean data for distillation 
train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=1024, shuffle=True)
val_loader   = DataLoader(TensorDataset(X_val,   y_val),   batch_size=1024, shuffle=False)
test_loader  = DataLoader(TensorDataset(X_test,  y_test),  batch_size=1024, shuffle=False)

# Training loop
for epoch in range(epochs):
    student.train()
    for t in teachers:
        t.eval()

    running_loss = 0.0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}")

    for inputs, labels in pbar:
        inputs = inputs.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        # Student forward
        student_logits = student(inputs)                                # (B, 1)

        # Teacher forwards (no grad needed)
        with torch.no_grad():
            teacher_logits_list = [t(inputs) for t in teachers]        # list of (B, 1)

        # Adaptive weights
        weights = compute_adaptive_weights(student_logits.detach(), teacher_logits_list)

        # Losses
        ce_loss   = criterion_ce(student_logits.squeeze(-1), labels.float())
        kd_loss   = distillation_loss(student_logits, teacher_logits_list, weights)
        total_loss = (1 - alpha) * ce_loss + alpha * kd_loss

        total_loss.backward()
        optimizer.step()

        running_loss += total_loss.item()
        pbar.set_postfix({'loss': f"{total_loss.item():.4f}",
                          'ce':   f"{ce_loss.item():.4f}",
                          'kd':   f"{kd_loss.item():.4f}"})

    avg_loss = running_loss / len(train_loader)

    # Validation
    student.eval()
    all_preds, all_labels_list = [], []
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            logits = student(inputs).squeeze(-1)
            preds  = (torch.sigmoid(logits) > 0.5).long()
            all_preds.extend(preds.cpu().numpy())
            all_labels_list.extend(labels.cpu().numpy())

    val_acc = accuracy_score(all_labels_list, all_preds) * 100
    val_f1  = f1_score(all_labels_list, all_preds)
    print(f"Epoch {epoch+1}: Loss {avg_loss:.4f} | Val Acc {val_acc:.2f}% | Val F1 {val_f1:.4f}")

torch.save(student.state_dict(), 'mtkd_adr_student.pth')
print("Student model saved.")

# Final test on clean data
student.eval()
all_preds, all_labels_list = [], []
with torch.no_grad():
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        logits = student(inputs).squeeze(-1)
        preds  = (torch.sigmoid(logits) > 0.5).long()
        all_preds.extend(preds.cpu().numpy())
        all_labels_list.extend(labels.cpu().numpy())

print("\n--- MTKD-ADR STUDENT TEST METRICS (Clean) ---")
print(classification_report(all_labels_list, all_preds, target_names=['Attack', 'Benign']))

Epoch 1/5: 100%|██████████| 10746/10746 [04:18<00:00, 41.62it/s, loss=0.0348, ce=0.0905, kd=0.0110]


Epoch 1: Loss 0.0524 | Val Acc 97.71% | Val F1 0.9840


Epoch 2/5: 100%|██████████| 10746/10746 [04:16<00:00, 41.88it/s, loss=0.0999, ce=0.1592, kd=0.0745]


Epoch 2: Loss 0.0389 | Val Acc 97.96% | Val F1 0.9859


Epoch 3/5: 100%|██████████| 10746/10746 [04:16<00:00, 41.95it/s, loss=0.0438, ce=0.1152, kd=0.0132]


Epoch 3: Loss 0.0373 | Val Acc 97.98% | Val F1 0.9860


Epoch 4/5: 100%|██████████| 10746/10746 [04:16<00:00, 41.82it/s, loss=0.0422, ce=0.0961, kd=0.0191]


Epoch 4: Loss 0.0367 | Val Acc 98.00% | Val F1 0.9862


Epoch 5/5: 100%|██████████| 10746/10746 [04:14<00:00, 42.15it/s, loss=0.0308, ce=0.0785, kd=0.0103]


Epoch 5: Loss 0.0358 | Val Acc 97.99% | Val F1 0.9861
Student model saved.

--- MTKD-ADR STUDENT TEST METRICS (Clean) ---
              precision    recall  f1-score   support

      Attack       0.99      0.94      0.96    274823
      Benign       0.98      1.00      0.99    687691

    accuracy                           0.98    962514
   macro avg       0.98      0.97      0.97    962514
weighted avg       0.98      0.98      0.98    962514



In [55]:
# def compute_adaptive_weights(teacher_logits_list):
#     weights = []
#     for t_logits in teacher_logits_list:
#         t_prob = torch.sigmoid(t_logits)
#         # confidence = how far from 0.5 the teacher is
#         confidence = torch.abs(t_prob - 0.5).mean()
#         weights.append(confidence)
    
#     total = sum(weights)
#     normalized = [w / total for w in weights]
#     return normalized

In [57]:
def generate_perturbation(inputs, labels, feature_indices, epsilon=None):
    if epsilon is None:
        epsilon = random.uniform(0.01, 0.1)
    
    inputs_perturbed = inputs.clone()
    is_malicious = (labels == 0)

    perturbation_mask = torch.zeros(inputs.shape[1], device=inputs.device)
    perturbation_mask[feature_indices] = 1.0

    if is_malicious.any():
        X_mal = inputs[is_malicious]
        y_mal = labels[is_malicious].long()

        _, adv_mal, _ = attack(fmodel, X_mal, y_mal, epsilons=epsilon)

        adv_mal = X_mal + ((adv_mal - X_mal) * perturbation_mask)
        adv_mal = torch.clamp(adv_mal, x_min, x_max)

        nonzero_mask = X_mal.abs() > 1e-2
        per_feature_mape = torch.abs((adv_mal - X_mal) / (X_mal.abs() + 1e-8)) * 100.0
        mape_ok = ((per_feature_mape <= mape_threshold) | ~nonzero_mask).all(dim=1)
        adv_mal[~mape_ok] = X_mal[~mape_ok]

        inputs_perturbed[is_malicious] = adv_mal

    return inputs_perturbed


# Student model + training setup
student = DDoSDetector(num_classes=1).to(device)
criterion_ce = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(student.parameters(), lr=1e-3, weight_decay=1e-5)

alpha = 0.7
epochs = 5

train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=1024, shuffle=True)
val_loader   = DataLoader(TensorDataset(X_val, y_val), batch_size=1024, shuffle=False)
test_loader  = DataLoader(TensorDataset(X_test, y_test), batch_size=1024, shuffle=False)

for epoch in range(epochs):
    student.train()
    for t in teachers:
        t.eval()

    running_loss = 0.0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}")

    for inputs, labels in pbar:
        inputs = inputs.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        
        inputs_perturbed = generate_perturbation(inputs, labels, feature_indices)   # still (B, F)

        # Student forward on clean data
        student_logits = student(inputs)                               # (B, 1)

        # Generate perturbed inputs for teachers
        with torch.no_grad():
            # All teachers see the perturbed input
            teacher_logits_list = [t(inputs_perturbed) for t in teachers]  # list of (B, 1)

        # Adaptive weights based on teacher confidence on perturbed input
        weights = compute_adaptive_weights(teacher_logits_list)

        # Losses
        ce_loss  = criterion_ce(student_logits.squeeze(-1), labels.float())
        kd_loss  = distillation_loss(student_logits, teacher_logits_list, weights)
        total_loss = (1 - alpha) * ce_loss + alpha * kd_loss

        total_loss.backward()
        optimizer.step()

        running_loss += total_loss.item()
        pbar.set_postfix({'loss': f"{total_loss.item():.4f}",
                          'ce':   f"{ce_loss.item():.4f}",
                          'kd':   f"{kd_loss.item():.4f}"})

    avg_loss = running_loss / len(train_loader)

    # Validation
    student.eval()
    all_preds, all_labels_list = [], []
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            logits = student(inputs).squeeze(-1)
            preds  = (torch.sigmoid(logits) > 0.5).long()
            all_preds.extend(preds.cpu().numpy())
            all_labels_list.extend(labels.cpu().numpy())

    val_acc = accuracy_score(all_labels_list, all_preds) * 100
    val_f1  = f1_score(all_labels_list, all_preds)
    print(f"Epoch {epoch+1}: Loss {avg_loss:.4f} | Val Acc {val_acc:.2f}% | Val F1 {val_f1:.4f}")

torch.save(student.state_dict(), 'mtkd_adr_student.pth')
print("Student model saved.")

Epoch 1/5: 100%|██████████| 10746/10746 [13:57<00:00, 12.83it/s, loss=0.1702, ce=0.0621, kd=0.2165]


Epoch 1: Loss 0.2368 | Val Acc 97.58% | Val F1 0.9831


Epoch 2/5: 100%|██████████| 10746/10746 [13:59<00:00, 12.81it/s, loss=0.2646, ce=0.1994, kd=0.2925]


Epoch 2: Loss 0.2074 | Val Acc 97.70% | Val F1 0.9840


Epoch 3/5: 100%|██████████| 10746/10746 [13:55<00:00, 12.86it/s, loss=0.4728, ce=0.1896, kd=0.5942]


Epoch 3: Loss 0.2042 | Val Acc 97.70% | Val F1 0.9840


Epoch 4/5: 100%|██████████| 10746/10746 [13:51<00:00, 12.92it/s, loss=0.2093, ce=0.1062, kd=0.2535]


Epoch 4: Loss 0.2022 | Val Acc 97.68% | Val F1 0.9838


Epoch 5/5: 100%|██████████| 10746/10746 [13:51<00:00, 12.92it/s, loss=0.1408, ce=0.0787, kd=0.1674]


Epoch 5: Loss 0.2004 | Val Acc 97.70% | Val F1 0.9840
Student model saved.


In [29]:
# Final test on clean data
student.eval()
all_preds, all_labels_list = [], []
with torch.no_grad():
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        logits = student(inputs).squeeze(-1)
        preds  = (torch.sigmoid(logits) > 0.3).long()
        all_preds.extend(preds.cpu().numpy())
        all_labels_list.extend(labels.cpu().numpy())

print("\n--- MTKD-ADR STUDENT TEST METRICS (Clean) ---")
print(classification_report(all_labels_list, all_preds, target_names=['Attack', 'Benign']))


--- MTKD-ADR STUDENT TEST METRICS (Clean) ---
              precision    recall  f1-score   support

      Attack       0.99      0.94      0.96    274823
      Benign       0.98      1.00      0.99    687691

    accuracy                           0.98    962514
   macro avg       0.98      0.97      0.97    962514
weighted avg       0.98      0.98      0.98    962514



In [24]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Get baseline model  
student_model = DDoSDetector(num_classes=1).to(device) 
student_model.load_state_dict(torch.load("mtkd_adr_student.pth", map_location=device))

<All keys matched successfully>

In [91]:
print("Benign (label=1):", (y_test == 1).sum().item())
print("Malicious (label=0):", (y_test == 0).sum().item())

Benign (label=1): 687691
Malicious (label=0): 274823


In [ ]:
def evaluate_under_attack(model, test_loader, attack, epsilon, device, 
                           feature_indices, mape_threshold=20.0):
    model.eval()
    all_preds = []
    all_labels = []

    wrapped = DDoSDetectorBinaryWrapper(model).to(device)
    wrapped.eval()
    fmodel = fb.PyTorchModel(wrapped, bounds=(X_test.min().item(), X_test.max().item()))

    for inputs, labels in tqdm(test_loader, desc=f"Evaluating under attack ε={epsilon}"):
        inputs, labels = inputs.to(device), labels.to(device)

        is_malicious = (labels == 0)
        adv_inputs = inputs.clone()

        if is_malicious.sum() > 0:
            X_mal = inputs[is_malicious]
            y_mal = labels[is_malicious]
            
            # generate adversarial examples
            _, adv_mal, _ = attack(fmodel, X_mal, y_mal, epsilons=epsilon)
            
            # feature mask
            mask = torch.zeros_like(X_mal)
            mask[:, feature_indices] = 1.0
            adv_mal = X_mal + ((adv_mal - X_mal) * mask)
            adv_mal = torch.clamp(adv_mal, X_test.min().item(), X_test.max().item())
            
            # # MAPE constraint 
            nonzero_mask = X_mal.abs() > 1e-2
            per_feature_mape = torch.abs((adv_mal - X_mal) / (X_mal.abs() + 1e-8)) * 100.0
            mape_ok = ((per_feature_mape <= mape_threshold) | ~nonzero_mask).all(dim=1)

            adv_mal[~mape_ok] = X_mal[~mape_ok]  
            
            # evaluate the full batch
            adv_inputs = inputs.clone()
            adv_inputs[is_malicious] = adv_mal_fixed
    
        with torch.no_grad():
            outputs = model(adv_inputs).squeeze(-1)
            probs = torch.sigmoid(outputs)
            preds = (probs > 0.5).long()

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    accuracy = accuracy_score(all_labels, all_preds) * 100
    f1 = f1_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds)
    recall = recall_score(all_labels, all_preds)

    return accuracy, f1, precision, recall

In [62]:
# 1. Gradients flow
wrapped_fixed = DDoSDetectorBinaryWrapper(model).cuda()
with torch.enable_grad():
    inputs_req = inputs[:4].cuda().clone().requires_grad_(True)
    out = wrapped_fixed(inputs_req)
    out.sum().backward()
    print("Gradient norm:", inputs_req.grad.norm().item())  # should be ~0.85

# 2. Predictions are still correct
with torch.no_grad():
    inputs_test = inputs[:4].cuda()
    out = wrapped_fixed(inputs_test)
    print("Output shape:", out.shape)          # should be (4, 2)
    print("Sample outputs:", out)              # class 1 logit should dominate for malicious
    preds = out.argmax(dim=1)
    print("Predictions:", preds)      

# 3. Foolbox can find adversarial examples
attack = FGSM()
model.eval()
fmodel = fb.PyTorchModel(wrapped_fixed, bounds=(0.0, 1.0))
X_test = inputs[:4].cuda()
y_test = labels[:4].cuda().long()
with torch.enable_grad():
    _, adv, _ = attack(fmodel, X_test, y_test, epsilons=0.1)
print("Max perturbation:", (adv - X_test).abs().max().item())  # should be > 0

Gradient norm: 349.2942199707031
Output shape: torch.Size([4, 2])
Sample outputs: tensor([[ 0.0000,  9.1612],
        [ 0.0000, -6.8038],
        [ 0.0000,  2.5893],
        [ 0.0000,  2.2316]], device='cuda:0')
Predictions: tensor([1, 0, 1, 1], device='cuda:0')
Max perturbation: 0.10000002384185791


/home/zawofeso/miniforge3/envs/IDS/lib/python3.10/site-packages/foolbox/models/pytorch.py:36: UserWarning: The PyTorch model is in training mode and therefore might not be deterministic. Call the eval() method to set it in evaluation mode if this is not intended.
  warnings.warn(


In [53]:
def set_bn_to_train_stats(model):
    """Force batchnorm to use batch statistics (not running stats) for gradient flow"""
    for module in model.modules():
        if isinstance(module, nn.BatchNorm1d):
            module.momentum = None  # use cumulative moving average
            module.train()          # use batch stats, not running stats

def evaluate_under_attack(model, test_loader, attack, epsilon, device):
    model.eval()
    all_preds = []
    all_labels = []

    # Fix BatchNorm for attack generation
    set_bn_to_train_stats(model)
    
    wrapped = DDoSDetectorBinaryWrapper(model).to(device)
    fmodel = fb.PyTorchModel(wrapped, bounds=(0.0, 1.0))

    for inputs, labels in tqdm(test_loader, desc=f"Evaluating under attack ε={epsilon}"):
        inputs, labels = inputs.to(device), labels.to(device)

        is_malicious = (labels == 0)
        adv_inputs = inputs.clone()

        if is_malicious.sum() > 0:
            X_mal = inputs[is_malicious]
            y_mal = labels[is_malicious]

            with torch.enable_grad():
                _, adv_mal, _ = attack(fmodel, X_mal, y_mal, epsilons=epsilon)
            
            adv_mal = torch.clamp(adv_mal, 0.0, 1.0)
            adv_inputs = inputs.clone()
            adv_inputs[is_malicious] = adv_mal

        # Restore eval mode for inference
        model.eval()

        with torch.no_grad():
            outputs = model(adv_inputs).squeeze(-1)
            probs = torch.sigmoid(outputs)
            preds = (probs > 0.5).long()

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    accuracy = accuracy_score(all_labels, all_preds) * 100
    f1 = f1_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds)
    recall = recall_score(all_labels, all_preds)

    return accuracy, f1, precision, recall

In [55]:
model

DDoSDetector(
  (cnn_backbone): Sequential(
    (0): Conv1d(1, 108, kernel_size=(5,), stride=(1,), padding=(2,))
    (1): BatchNorm1d(108, eps=1e-05, momentum=None, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Conv1d(108, 108, kernel_size=(5,), stride=(1,), padding=(2,))
    (4): BatchNorm1d(108, eps=1e-05, momentum=None, affine=True, track_running_stats=True)
    (5): ReLU()
    (6): Conv1d(108, 108, kernel_size=(5,), stride=(1,), padding=(2,))
    (7): BatchNorm1d(108, eps=1e-05, momentum=None, affine=True, track_running_stats=True)
    (8): ReLU()
    (9): Conv1d(108, 108, kernel_size=(5,), stride=(1,), padding=(2,))
    (10): BatchNorm1d(108, eps=1e-05, momentum=None, affine=True, track_running_stats=True)
    (11): ReLU()
    (12): Conv1d(108, 108, kernel_size=(5,), stride=(1,), padding=(2,))
    (13): BatchNorm1d(108, eps=1e-05, momentum=None, affine=True, track_running_stats=True)
    (14): ReLU()
    (15): Conv1d(108, 108, kernel_size=(5,), stride=(1,), paddi

In [25]:
wrapped = DDoSDetectorBinaryWrapper(model).to(device)

# Test a single batch
inputs, labels = next(iter(test_loader))
inputs, labels = inputs.to(device), labels.to(device)

inputs.requires_grad = True
out = inputs
print("Wrapper output shape:", out.shape)
print("Wrapper output sample:", out[:5])
loss = out.sum()
loss.backward()
print("Input gradient norm:", inputs.grad.norm().item())

Wrapper output shape: torch.Size([2048, 74])
Wrapper output sample: tensor([[2.8163e-06, 0.0000e+00, 7.1429e-02, 2.8955e-02, 1.5087e-02, 4.3850e-02,
         2.8082e-02, 2.1925e-01, 0.0000e+00, 3.9860e-02, 3.9041e-02, 1.7906e-01,
         0.0000e+00, 7.5224e-06, 0.0000e+00, 4.4746e-06, 7.7301e-06, 0.0000e+00,
         0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00,
         0.0000e+00, 0.0000e+00, 0.0000e+00, 2.5641e-02, 2.4096e-02, 2.8082e-02,
         3.9860e-02, 2.2958e-01, 2.3200e-02, 5.3824e-04, 1.0000e+00, 0.0000e+00,
         2.8955e-02, 7.1429e-02, 1.5087e-02, 0.0000e+00, 0.0000e+00, 0.0000e+00,
         1.6667e-01, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00,
         0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 1.0000e+00,
         1.0000e+00, 0.0000e+00, 1.0000e+00, 0.0000e+00, 1.0000e+00, 0.0000e+00,
         1.0000e+00, 0.0000e+00, 1.0000e+00, 0.0000e+00, 1.0000e+00, 0.0000e+00,
         1.0000e+00, 0.0000e+00, 1.0000e+

In [ ]:
class DDoSDetectorBinaryWrapper(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model
    
    def forward(self, x):
        logit = self.model(x)  # (B, 1)
        zeros = torch.zeros_like(logit)
        return torch.cat([zeros, logit], dim=1)  # (B, 2)

wrapped_fixed = DDoSDetectorBinaryWrapper(model_test).cpu()

with torch.enable_grad():
    inputs_req = inputs[:4].cpu().clone().requires_grad_(True)
    out = wrapped_fixed(inputs_req)
    out.sum().backward()
    print("Fixed wrapper gradient norm:", inputs_req.grad.norm().item())

Fixed wrapper gradient norm: 0.8579237461090088


In [46]:
# Test 1: layer by layer without wrapper
model_test = DDoSDetector(num_classes=1).cpu()

with torch.enable_grad():
    inputs_req = inputs[:4].cpu().clone().requires_grad_(True)
    x = inputs_req.unsqueeze(1)
    
    for i, layer in enumerate(model_test.cnn_backbone):
        x = layer(x)
        inputs_req.grad = None
        x.sum().backward(retain_graph=True)
        grad = inputs_req.grad.norm().item() if inputs_req.grad is not None else None
        print(f"Layer {i} ({layer.__class__.__name__}): grad={grad}")

Layer 0 (Conv1d): grad=52.9182014465332
Layer 1 (BatchNorm1d): grad=1.902133590192534e-05
Layer 2 (ReLU): grad=613.2073974609375
Layer 3 (Conv1d): grad=76.44561767578125
Layer 4 (BatchNorm1d): grad=7.2701077442616224e-06
Layer 5 (ReLU): grad=527.7975463867188
Layer 6 (Conv1d): grad=104.38549041748047
Layer 7 (BatchNorm1d): grad=2.0266488718334585e-05
Layer 8 (ReLU): grad=590.5499267578125
Layer 9 (Conv1d): grad=159.9191131591797
Layer 10 (BatchNorm1d): grad=1.3621551261167042e-05
Layer 11 (ReLU): grad=730.20556640625
Layer 12 (Conv1d): grad=225.41879272460938
Layer 13 (BatchNorm1d): grad=1.7980091797653586e-05
Layer 14 (ReLU): grad=787.954833984375
Layer 15 (Conv1d): grad=258.0405578613281
Layer 16 (BatchNorm1d): grad=2.3424623577739112e-05
Layer 17 (ReLU): grad=898.7491455078125
Layer 18 (Conv1d): grad=331.84716796875
Layer 19 (BatchNorm1d): grad=3.176033715135418e-05
Layer 20 (ReLU): grad=965.7498779296875
Layer 21 (Conv1d): grad=348.57080078125
Layer 22 (BatchNorm1d): grad=3.0280372

In [50]:
# Wrapper to convert binary classifier output to 2-class logits for foolbox
class DDoSDetectorBinaryWrapper(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model
    
    def forward(self, x):
        logit = self.model(x)  # (B, 1)
        negative_logit = -logit
        return torch.cat([negative_logit, logit], dim=1)  # (B, 2)


class FixedDDoSDetectorBinaryWrapper(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model
    
    def forward(self, x):
        logit = self.model(x)
        zeros = torch.zeros_like(logit)
        return torch.cat([zeros, logit], dim=1)

In [52]:
# Test with fresh model vs loaded model
model = DDoSDetector(num_classes=1).cpu()
wrapped_fixed = FixedDDoSDetectorBinaryWrapper(model)

with torch.enable_grad():
    inputs_req = inputs[:4].cpu().clone().requires_grad_(True)
    out = wrapped_fixed(inputs_req)
    out.sum().backward()
    print("Fixed model gradient norm:", inputs_req.grad.norm().item())

Fixed model gradient norm: 0.9325132966041565


In [53]:
# Test with fresh model vs loaded model
model = DDoSDetector(num_classes=1).cpu()
wrapped_fixed = DDoSDetectorBinaryWrapper(model)

with torch.enable_grad():
    inputs_req = inputs[:4].cpu().clone().requires_grad_(True)
    out = wrapped_fixed(inputs_req)
    out.sum().backward()
    print("Fixed model gradient norm:", inputs_req.grad.norm().item())

Fixed model gradient norm: 0.0


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Get baseline model  
pgd_teacher_model = DDoSDetector(num_classes=1).to(device) 
pgd_teacher_model.load_state_dict(torch.load("baseline_nids_cnn.pth", map_location=device))

In [ ]:
def evaluate_under_attack(model, test_loader, attack, epsilon, device):
    model.eval()
    all_preds = []
    all_labels = []

    wrapped = DDoSDetectorBinaryWrapper(model).to(device)
    wrapped.eval()
    fmodel = fb.PyTorchModel(wrapped, bounds=(X_test.min().item(), X_test.max().item()))
    print("X_test min:", X_test.min().item())
    print("X_test max:", X_test.max().item())

    for inputs, labels in tqdm(test_loader, desc=f"Evaluating under attack ε={epsilon}"):
        inputs, labels = inputs.to(device), labels.to(device)

        is_malicious = (labels == 0)
        adv_inputs = inputs.clone()

        if is_malicious.sum() > 0:
            X_mal = inputs[is_malicious]
            y_mal = labels[is_malicious]
            
            _, adv_mal, _ = attack(fmodel, X_mal, y_mal, epsilons=epsilon)
            adv_mal = torch.clamp(adv_mal, X_test.min().item(), X_test.max().item())

            adv_inputs = inputs.clone()
            adv_inputs[is_malicious] = adv_mal
    
        with torch.no_grad():
            outputs = model(adv_inputs).squeeze(-1)
            probs = torch.sigmoid(outputs)
            preds = (probs > 0.5).long()

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    accuracy = accuracy_score(all_labels, all_preds) * 100
    f1 = f1_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds)
    recall = recall_score(all_labels, all_preds)

    return accuracy, f1, precision, recall

In [39]:
# features_to_peturb = ['Flow Duration', 'Flow IAT Mean', 'Flow IAT Max', 'Flow IAT Min', 'Flow IAT Std',
#                       'Fwd IAT Tot', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min',
#                       'Bwd IAT Tot', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min',
#                       'Active Mean', 'Active Std', 'Active Max', 'Active Min',
#                       'Idle Mean', 'Idle Std', 'Idle Max', 'Idle Min'   
# ]

# feature_indices = [numerical_cols.index(feature) for feature in features_to_peturb]

# attack_list = [(FGSM(), "FGSM"), (LinfBasicIterativeAttack(abs_stepsize=0.01, steps=20), "IFGSM"), (PGD(rel_stepsize=0.02, steps=40, random_start=True), "PGD")]
attack = FGSM()

for epsilon in [0.1, 0.2, 0.3]:
    accuracy, f1, precision, recall = evaluate_under_attack(pgd_teacher_model, test_loader, attack, epsilon, device)
    print(f"\n ε={epsilon}")
    print(f"Accuracy:  {accuracy:.2f}%")
    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")
    print(f"F1 Score:  {f1:.4f}")

Evaluating under attack ε=0.1:  13%|█▎        | 126/940 [00:02<00:21, 37.14it/s]

Evaluating under attack ε=0.1: 100%|██████████| 940/940 [00:20<00:00, 45.50it/s]



 ε=0.1
Accuracy:  70.75%
Precision: 0.7126
Recall:    0.9897
F1 Score:  0.8286


Evaluating under attack ε=0.2: 100%|██████████| 940/940 [00:20<00:00, 45.08it/s]



 ε=0.2
Accuracy:  70.74%
Precision: 0.7126
Recall:    0.9897
F1 Score:  0.8286


Evaluating under attack ε=0.3: 100%|██████████| 940/940 [00:19<00:00, 48.65it/s]



 ε=0.3
Accuracy:  70.72%
Precision: 0.7124
Recall:    0.9897
F1 Score:  0.8285


In [ ]:
def evaluate_under_attack(model, test_loader, attack, epsilon, device):
    model.eval()
    all_preds = []
    all_labels = []

    # wrap model for foolbox
    wrapped = DDoSDetectorBinaryWrapper(model).to(device)
    wrapped.eval()
    fmodel = fb.PyTorchModel(wrapped, bounds=(X_test.min().item(), X_test.max().item()))

    for inputs, labels in tqdm(test_loader, desc=f"Evaluating under attack ε={epsilon}"):
        inputs, labels = inputs.to(device), labels.to(device)

        # generate adversarial examples
        _, adv_inputs, _ = attack(fmodel, inputs, labels, epsilons=epsilon)

        # evaluate model on adversarial inputs
        with torch.no_grad():
            outputs = model(adv_inputs).squeeze(1)
            probs = torch.sigmoid(outputs)
            preds = (probs > 0.5).long()

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    accuracy = accuracy_score(all_labels, all_preds) * 100
    f1 = f1_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds)
    recall = recall_score(all_labels, all_preds)

    return accuracy, f1, precision, recall


    def evaluate_under_attack(model, test_loader, attack, epsilon, device, 
                           feature_indices, mape_threshold=20.0):
    model.eval()
    all_preds = []
    all_labels = []

    wrapped = DDoSDetectorBinaryWrapper(model).to(device)
    wrapped.eval()
    fmodel = fb.PyTorchModel(wrapped, bounds=(X_test.min().item(), X_test.max().item()))

    for inputs, labels in tqdm(test_loader, desc=f"Evaluating under attack ε={epsilon}"):
        inputs, labels = inputs.to(device), labels.to(device)

        is_malicious = (labels == 0)
        adv_inputs = inputs.clone()

        if is_malicious.sum() > 0:
            X_mal = inputs[is_malicious]
            y_mal = labels[is_malicious]
            
            # generate adversarial examples
            _, adv_mal, _ = attack(fmodel, X_mal, y_mal, epsilons=epsilon)
            
            # feature mask
            mask = torch.zeros_like(X_mal)
            mask[:, feature_indices] = 1.0
            adv_mal = X_mal + ((adv_mal - X_mal) * mask)
            adv_mal = torch.clamp(adv_mal, X_test.min().item(), X_test.max().item())
            
            # # MAPE constraint 
            nonzero_mask = X_mal.abs() > 1e-2
            per_feature_mape = torch.abs((adv_mal - X_mal) / (X_mal.abs() + 1e-8)) * 100.0
            mape_ok = ((per_feature_mape <= mape_threshold) | ~nonzero_mask).all(dim=1)
            adv_mal = adv_mal[mape_ok]
            y_mal = y_mal[mape_ok]
            
            # only evaluate on MAPE passing malicious samples + all benign
            adv_inputs = torch.cat([inputs[~is_malicious], adv_mal], dim=0)
            labels = torch.cat([labels[~is_malicious], y_mal], dim=0)
    
        with torch.no_grad():
            outputs = model(adv_inputs).squeeze(-1)
            probs = torch.sigmoid(outputs)
            preds = (probs > 0.5).long()

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    accuracy = accuracy_score(all_labels, all_preds) * 100
    f1 = f1_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds)
    recall = recall_score(all_labels, all_preds)

    return accuracy, f1, precision, recall


    features_to_peturb = ['Flow Duration', 'Flow IAT Mean', 'Flow IAT Max', 'Flow IAT Min', 'Flow IAT Std',
                      'Fwd IAT Tot', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min',
                      'Bwd IAT Tot', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min',
                      'Active Mean', 'Active Std', 'Active Max', 'Active Min',
                      'Idle Mean', 'Idle Std', 'Idle Max', 'Idle Min'   
]

feature_indices = [numerical_cols.index(feature) for feature in features_to_peturb]

# attack_list = [(FGSM(), "FGSM"), (LinfBasicIterativeAttack(abs_stepsize=0.01, steps=20), "IFGSM"), (PGD(rel_stepsize=0.02, steps=40, random_start=True), "PGD")]
attack = PGD(rel_stepsize=0.02, steps=40, random_start=True)

for epsilon in [0.1, 0.2, 0.3]:
    accuracy, f1, precision, recall = evaluate_under_attack(pgd_teacher_model, test_loader, attack, epsilon, device, feature_indices)
    print(f"\n ε={epsilon}")
    print(f"Accuracy:  {accuracy:.2f}%")
    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")
    print(f"F1 Score:  {f1:.4f}")

In [ ]:
attack = FGSM()

for epsilon in [0.1, 0.2, 0.3]:
    acc, f1, prec, rec = evaluate_under_attack(
        model, test_loader, attack, epsilon, device
    )
    print(f"ε={epsilon} | Acc: {acc:.2f}% | F1: {f1:.4f} | Precision: {prec:.4f} | Recall: {rec:.4f}")

### PGD Teacher Model

In [16]:
# Initialize clone of baseline model for adversarial training
fgsm_teacher_model = DDoSDetector(num_classes=1)

In [17]:
def load_adversarial_data(adv_data_path, device='cuda'):
    adv_data = torch.load(adv_data_path, map_location=device)

    X_adv, y_adv = adv_data
    print(f"Loaded adversarial data from tuple")
    print(f" X_adv shape: {X_adv.shape}")
    print(f" y_adv shape: {y_adv.shape}")
    
    return X_adv, y_adv

In [18]:
# Load adversarial examples from .pt file
X_fgsm_adv, y_fgsm_adv = load_adversarial_data('fgsm_augmented_dataset.pt')

Loaded adversarial data from tuple
 X_adv shape: torch.Size([11003915, 74])
 y_adv shape: torch.Size([11003915])


In [19]:
# Confirm dtype is float32 for features and labels
print(f"X_fgsm_adv dtype: {X_fgsm_adv.dtype}")
print(f"y_fgsm_adv dtype: {y_fgsm_adv.dtype}")

X_fgsm_adv dtype: torch.float32
y_fgsm_adv dtype: torch.float32


In [20]:
def prepare_adversarial_loaders(X_adv, y_adv, batch_size=256, 
                                train_ratio=0.8):
    
    # Create dataset
    dataset = TensorDataset(X_adv, y_adv)
    
    # Calculate split sizes
    total_size = len(dataset)
    train_size = int(train_ratio * total_size)
    val_size = total_size - train_size
    
    # Split dataset
    train_dataset, val_dataset = random_split(
        dataset, 
        [train_size, val_size],
        generator=torch.Generator().manual_seed(42)  
    ) 
    
    # Create DataLoaders
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    
    print(f"Created DataLoaders:")
    print(f" Train: {len(train_dataset)} samples")
    print(f" Val:   {len(val_dataset)} samples")
    
    return train_loader, val_loader

In [21]:
# def prepare_adversarial_loaders(X_adv, y_adv, batch_size=256, 
#                                 train_ratio=0.8):
    
#     # Create dataset
#     dataset = TensorDataset(X_adv, y_adv)
    
#     # Calculate split sizes
#     total_size = len(dataset)
#     train_size = int(train_ratio * total_size)
#     val_size = total_size - train_size
    
#     # Split dataset
#     train_dataset, val_dataset = random_split(
#         dataset, 
#         [train_size, val_size],
#         generator=torch.Generator().manual_seed(42)  
#     ) 
    
#     # Create DataLoaders
#     train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
#     val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    
#     print(f"Created DataLoaders:")
#     print(f" Train: {len(train_dataset)} samples")
#     print(f" Val:   {len(val_dataset)} samples")
    
#     return train_loader, val_loader

In [22]:
train_fgsm_loader, val_fgsm_loader = prepare_adversarial_loaders(
    X_fgsm_adv, y_fgsm_adv,  
    batch_size=256,
    train_ratio=0.8)

Created DataLoaders:
 Train: 8803132 samples
 Val:   2200783 samples


In [ ]:
# Hyperparameters
num_classes = 1  
learning_rate = 1e-3
l2_lambda = 1e-5
l1_lambda = 1e-6
epochs = 5

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Initialize clone of baseline model for adversarial training
model = DDoSDetector(num_classes=num_classes).to(device)

# Loss and Optimizer
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=l2_lambda)

# Training and Validation Loop
for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    
    # Wrap the loader in tqdm for progress bar
    pbar = tqdm(train_fgsm_loader, desc=f"Epoch {epoch+1}")
    
    for inputs, labels in pbar:
        inputs, labels = inputs.to(device, non_blocking=True), labels.to(device, non_blocking=True)

        optimizer.zero_grad()
        outputs = model(inputs).squeeze(1)  # Now (batch,) shape
        loss = criterion(outputs, labels.float())  

        # L1 Regularization
        l1_reg = torch.tensor(0., device=device)
        for p in model.parameters():
            l1_reg += p.abs().sum() 
        
        loss = loss + l1_lambda * l1_reg

        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        
        # Update the progress bar with the current loss every batch
        pbar.set_postfix({'loss': f"{loss.item():.4f}"})

    avg_train_loss = running_loss / len(train_loader)

    # --- VALIDATION PHASE---
    model.eval()
    val_running_loss = 0.0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for inputs, labels in val_fgsm_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            outputs = model(inputs).squeeze(1)
            # Calculate validation loss
            v_loss = criterion(outputs, labels.float())  # ← ADDED .float()
            val_running_loss += v_loss.item()

            v_probs = torch.sigmoid(outputs)
            v_predicted = (v_probs > 0.5).long()

            all_preds.extend(v_predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    # Calculate metrics
    avg_val_loss = val_running_loss / len(val_loader)
    val_accuracy = accuracy_score(all_labels, all_preds) * 100

    print(f"Epoch {epoch+1}: Train Loss {avg_train_loss:.4f}, Val Loss {avg_val_loss:.4f}, Val Acc: {val_accuracy:.2f}%")


# Save the baseline model weights
model_path = 'fgsm_teacher_model_new.pth'
torch.save(model.state_dict(), model_path)
print(f"Model baseline saved to {model_path}")

# FINAL TEST PHASE
model.eval()

# Lists to store all predictions and actual labels
all_preds = []
all_labels = []

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        outputs = model(inputs).squeeze(1)
        probs = torch.sigmoid(outputs)
        predictions = (probs>0.5).long()

        all_preds.extend(predictions.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

test_accuracy = accuracy_score(all_labels, all_preds) * 100
test_precision = precision_score(all_labels, all_preds)
test_recall = recall_score(all_labels, all_preds)
test_f1 = f1_score(all_labels, all_preds)

print("\n--- TEST METRICS ---")
print(f"Accuracy: {test_accuracy:.2f}%")
print(f"Precision: {test_precision:.4f}")
print(f"Recall: {test_recall:.4f}")
print(f"F1 Score: {test_f1:.4f}")

Epoch 1: 100%|██████████| 34388/34388 [08:26<00:00, 67.89it/s, loss=0.0488]


Epoch 1: Train Loss 0.4699, Val Loss 1.1615, Val Acc: 98.27%


Epoch 2: 100%|██████████| 34388/34388 [07:57<00:00, 71.97it/s, loss=0.0213]


Epoch 2: Train Loss 0.4151, Val Loss 1.1380, Val Acc: 98.28%


Epoch 3: 100%|██████████| 34388/34388 [06:41<00:00, 85.57it/s, loss=0.0701]


Epoch 3: Train Loss 0.4062, Val Loss 1.1243, Val Acc: 98.32%


Epoch 4: 100%|██████████| 34388/34388 [06:52<00:00, 83.46it/s, loss=0.0250]


Epoch 4: Train Loss 0.4006, Val Loss 1.1238, Val Acc: 98.32%


Epoch 5: 100%|██████████| 34388/34388 [06:54<00:00, 82.91it/s, loss=0.1762]


Epoch 5: Train Loss 0.3971, Val Loss 1.2230, Val Acc: 98.22%
Model baseline saved to fgsm_teacher_model_new.pth

--- TEST METRICS ---
Accuracy: 97.91%
Precision: 0.9759
Recall: 0.9953
F1 Score: 0.9855


#### Evaluation

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Get baseline model
fgsm_teacher_model = DDoSDetector(num_classes=1).to(device)
fgsm_teacher_model.load_state_dict(torch.load("fgsm_teacher_model.pth", map_location=device))

In [ ]:
features_to_peturb = ['Flow Duration', 'Flow IAT Mean', 'Flow IAT Max', 'Flow IAT Min', 'Flow IAT Std',
                      'Fwd IAT Tot', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min',
                      'Bwd IAT Tot', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min',
                      'Active Mean', 'Active Std', 'Active Max', 'Active Min',
                      'Idle Mean', 'Idle Std', 'Idle Max', 'Idle Min'   
]
feature_indices = [numerical_cols.index(feature) for feature in features_to_peturb]

# Shuffle and Split indices
indices = torch.randperm(len(X_train))
split_point = len(X_train) // 2

clean_idx = indices[:split_point]
perturb_idx = indices[split_point:]

# Get the halves
X_clean, y_clean = X_train[clean_idx], y_train[clean_idx]
X_to_perturb, y_to_perturb = X_train[perturb_idx], y_train[perturb_idx]

# Generate Adversarial Samples 
is_malicious = (y_to_perturb == 0)

results, sample_dict = fgsm_attack_foolbox(fgsm_teacher_model, X_to_perturb[is_malicious],
                                            y_to_perturb[is_malicious].long(), 
                                            epsilon_values=[0.05, 0.07, 0.1], 
                                            batch_size=8192, 
                                            feature_indices=feature_indices)

### PGD Teacher Model

#### Evaluation

In [ ]:
features_to_peturb = ['Flow Duration', 'Flow IAT Mean', 'Flow IAT Max', 'Flow IAT Min', 'Flow IAT Std',
                      'Fwd IAT Tot', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min',
                      'Bwd IAT Tot', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min',
                      'Active Mean', 'Active Std', 'Active Max', 'Active Min',
                      'Idle Mean', 'Idle Std', 'Idle Max', 'Idle Min'   
]
feature_indices = [numerical_cols.index(feature) for feature in features_to_peturb]

# Shuffle and Split indices
indices = torch.randperm(len(X_train))
split_point = len(X_train) // 2

clean_idx = indices[:split_point]
perturb_idx = indices[split_point:]

# Get the halves
X_clean, y_clean = X_train[clean_idx], y_train[clean_idx]
X_to_perturb, y_to_perturb = X_train[perturb_idx], y_train[perturb_idx]

# Generate Adversarial Samples 
is_malicious = (y_to_perturb == 0)

results, sample_dict = pgd_attack_foolbox(fgsm_teacher_model, X_to_perturb[is_malicious],
                                            y_to_perturb[is_malicious].long(), 
                                            epsilon_values=[0.05, 0.07, 0.1], 
                                            batch_size=8192, 
                                            feature_indices=feature_indices)

In [36]:
def adversarial_finetuning(
    model,
    train_loader=None,
    val_loader=None,
    epochs=5,
    learning_rate=1e-4,  
    device='cuda'
):
    
    model = model.to(device)
    
    # Loss and optimizer
    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    
    # Finetuning loop
    print(f"\n{'='*60}")
    print("Starting Adversarial Finetuning for FGSM...")
    print(f"{'='*60}")
    
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}")
        
        for inputs, labels in pbar:
            inputs = inputs.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            
            optimizer.zero_grad()
            outputs = model(inputs).squeeze(1)
            loss = criterion(outputs, labels.float())
            
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
            pbar.set_postfix({'loss': f"{loss.item():.4f}"})
        
        avg_train_loss = running_loss / len(train_loader)
        
        # Validation
        if val_loader is not None:
            model.eval()
            val_loss = 0.0
            all_preds = []
            all_labels = []
            
            with torch.no_grad():
                for inputs, labels in val_loader:
                    inputs = inputs.to(device)
                    labels = labels.to(device)
                    
                    outputs = model(inputs).squeeze(1)
                    loss = criterion(outputs, labels.float())
                    val_loss += loss.item()
                    
                    probs = torch.sigmoid(outputs)
                    preds = (probs > 0.5).long()
                    
                    all_preds.extend(preds.cpu().numpy())
                    all_labels.extend(labels.cpu().numpy())
            
            avg_val_loss = val_loss / len(val_loader)
            val_acc = accuracy_score(all_labels, all_preds) * 100
            
            print(f"Epoch {epoch+1}: Train Loss {avg_train_loss:.4f}, "
                  f"Val Loss {avg_val_loss:.4f}, Val Acc {val_acc:.2f}%")
        else:
            print(f"Epoch {epoch+1}: Train Loss {avg_train_loss:.4f}")
    
    print(f"\n{'='*60}")
    print("Adversarial Finetuning Complete!")
    print(f"{'='*60}")
    
    return model

In [37]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

fgsm_finetuned_model = adversarial_finetuning(
    baseline_model,
    train_loader=train_fgsm_loader,
    val_loader=val_fgsm_loader,
    epochs=5,
    learning_rate=1e-4,  # Lower LR for finetuning
    device=device
    )


Starting Adversarial Finetuning for FGSM...


Epoch 1/5: 100%|██████████| 37508/37508 [06:33<00:00, 95.35it/s, loss=0.0364] 


Epoch 1: Train Loss 0.0562, Val Loss 0.0552, Val Acc 98.47%


Epoch 2/5: 100%|██████████| 37508/37508 [03:59<00:00, 156.30it/s, loss=0.0172]


Epoch 2: Train Loss 0.0547, Val Loss 0.0544, Val Acc 98.49%


...

Epoch 3: Train Loss 0.0544, Val Loss 0.0539, Val Acc 98.49%


Epoch 4/5: 100%|██████████| 37508/37508 [03:53<00:00, 160.59it/s, loss=0.0425]


Epoch 4: Train Loss 0.0542, Val Loss 0.0538, Val Acc 98.50%


Epoch 5/5: 100%|██████████| 37508/37508 [03:55<00:00, 159.61it/s, loss=0.0729]


Epoch 5: Train Loss 0.0539, Val Loss 0.0538, Val Acc 98.49%

Adversarial Finetuning Complete!


In [38]:
# Store model finetuned on FGSM adversarial data
model_path = 'FGSM/finetuned_fgsm_model.pth'
torch.save(fgsm_finetuned_model.state_dict(), model_path)
print(f"Adversarially fine-tuned model saved to {model_path}")

Adversarially fine-tuned model saved to FGSM/finetuned_fgsm_model.pth


In [39]:
def evaluate_model(model, test_loader, device='cuda'):
    model.eval()
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs = inputs.to(device)
            labels = labels.to(device)
            
            outputs = model(inputs).squeeze(1)
            probs = torch.sigmoid(outputs)
            preds = (probs > 0.5).long()
            
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    accuracy = accuracy_score(all_labels, all_preds) * 100
    precision = precision_score(all_labels, all_preds)
    recall = recall_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds)
    
    print(f"\n{'='*60}")
    print("Test Metrics")
    print(f"{'='*60}")
    print(f"Accuracy:  {accuracy:.2f}%")
    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")
    print(f"F1 Score:  {f1:.4f}")
    print(f"{'='*60}")
    
    return {'accuracy': accuracy, 'precision': precision, 'recall': recall, 'f1': f1}

In [40]:
# Evaluate finetuned model on clean set
test_dataset = TensorDataset(X_test, y_test)
test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False)
results = evaluate_model(fgsm_finetuned_model, test_loader, device)


Test Metrics
Accuracy:  98.10%
Precision: 0.9753
Recall:    0.9987
F1 Score:  0.9869


In [43]:
results_df = pd.DataFrame([results])
results_df.to_csv('FGSM/finetuned_fgsm_test_results.csv', index=False)
print("Finetuned model test results saved to 'FGSM/finetuned_fgsm_test_results.csv'")

Finetuned model test results saved to 'FGSM/finetuned_fgsm_test_results.csv'


### Blackbox Test FGSM Model

In [34]:
def black_box_evaluation(hardened_model, X_to_perturb, y_to_perturb, epsilons=[0.05, 0.1, 0.2], batch_size=4096):
    device = next(hardened_model.parameters()).device
    hardened_model.eval()

    wrapped_model = DDoSDetectorBinaryWrapper(hardened_model).to(device)
    wrapped_model.eval()

    fmodel = fb.PyTorchModel(wrapped_model, bounds=(X_to_perturb.min().item(), X_to_perturb.max().item()))
    attack = fb.attacks.HopSkipJumpAttack(steps=10)

    num_batches = (len(X_to_perturb) + batch_size - 1) // batch_size
    
    attacked_original_probs = []         
    all_adv_probs   = {eps: [] for eps in epsilons}
    all_success     = {eps: [] for eps in epsilons}

    all_original_preds = []
    all_labels         = []

    for batch_idx in range(num_batches):
        start_idx = batch_idx * batch_size
        end_idx   = min((batch_idx + 1) * batch_size, len(X_to_perturb))

        X_batch = X_to_perturb[start_idx:end_idx].to(device)
        y_batch = y_to_perturb[start_idx:end_idx].to(device)

        with torch.no_grad():
            original_outputs = hardened_model(X_batch)
            original_probs   = torch.sigmoid(original_outputs).squeeze(-1)  # (B,)
            original_preds   = (original_probs > 0.5).long()

        all_original_preds.append(original_preds.cpu())
        all_labels.append(y_batch.cpu())

        # Only attack samples the model currently gets RIGHT and are malicious (label=0)
        correct_malicious_mask = (y_batch == 0) & (original_preds == 0)

        if correct_malicious_mask.sum() == 0:
            continue

        X_targets = X_batch[correct_malicious_mask]
        y_targets  = y_batch[correct_malicious_mask].long()
        orig_probs_targets = original_probs[correct_malicious_mask]  # probs for attacked subset only

        # Run Square Attack across all epsilons at once
        _, clipped_adv, success = attack(fmodel, X_targets, y_targets, epsilons=epsilons)

        # Store original probs for this attacked subset (aligned with adv results)
        attacked_original_probs.append(orig_probs_targets.cpu())

        for i, eps in enumerate(epsilons):
            with torch.no_grad():
                adv_outputs = hardened_model(clipped_adv[i])
                adv_probs   = torch.sigmoid(adv_outputs).squeeze(-1)  

            all_adv_probs[eps].append(adv_probs.cpu())
            all_success[eps].append(success[i].cpu())

        del X_batch, y_batch, X_targets, y_targets, clipped_adv
        torch.cuda.empty_cache()

        if (batch_idx + 1) % 10 == 0:
            print(f"Processed {end_idx}/{len(X_to_perturb)} samples...")

    # Concatenate across all batches
    if len(attacked_original_probs) == 0:
        print("No correct malicious samples found to attack.")
        return {}

    orig_probs_cat = torch.cat(attacked_original_probs)  # (n_attacked,)

    print(f"\n{'='*50}")
    print("Black-Box Evaluation (HopSkipJumpAttack)")
    print(f"Total samples evaluated:  {len(X_to_perturb)}")
    print(f"Samples eligible for attack (correct malicious): {orig_probs_cat.shape[0]}")
    print(f"{'='*50}")

    results = {}

    for eps in epsilons:
        success_tensor   = torch.cat(all_success[eps])    # (n_attacked,) bool
        adv_probs_tensor = torch.cat(all_adv_probs[eps])  # (n_attacked,) float

        # These three tensors are all length n_attacked and fully aligned
        assert success_tensor.shape == adv_probs_tensor.shape == orig_probs_cat.shape, \
            f"Shape mismatch at eps={eps}: {success_tensor.shape}, {adv_probs_tensor.shape}, {orig_probs_cat.shape}"

        n_total    = success_tensor.shape[0]
        n_fooled   = success_tensor.sum().item()
        asr        = success_tensor.float().mean().item()
        robustness = 1.0 - asr

        # Confidence drop: how much did the model's malicious confidence drop for attacked samples
        # For malicious class (label=0), lower prob = model is less sure it's malicious = attack worked
        confidence_drop = (orig_probs_cat - adv_probs_tensor).mean().item()  # signed: positive = drop toward benign
        confidence_drop_abs = (orig_probs_cat - adv_probs_tensor).abs().mean().item()

        results[eps] = {
            "robustness":            robustness,
            "asr":                   asr,
            "n_fooled":              n_fooled,
            "n_total":               n_total,
            "confidence_drop":       confidence_drop,
            "confidence_drop_abs":   confidence_drop_abs,
        }

        print(f"\nepsilon={eps}:")
        print(f"  Robustness:            {robustness:.2%}")
        print(f"  Attack Success Rate:   {asr:.2%}")
        print(f"  Samples fooled:        {n_fooled} / {n_total}")
        print(f"  Confidence Drop (signed): {confidence_drop:.4f}")
        print(f"  Confidence Drop (abs):    {confidence_drop_abs:.4f}")

    print(f"{'='*50}\n")

    return results

In [35]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
fgsm_hardened_model = DDoSDetector(num_classes=1).to(device)
fgsm_hardened_model.load_state_dict(torch.load("FGSM/finetuned_fgsm_model.pth", map_location=device))

results, adv_examples = black_box_evaluation(fgsm_hardened_model, X_test[:10000], y_test[:10000], epsilons=[0.05, 0.1, 0.2], batch_size=64)

Processed 640/10000 samples...
Processed 1280/10000 samples...
Processed 1920/10000 samples...
Processed 2560/10000 samples...
Processed 3200/10000 samples...
Processed 3840/10000 samples...
Processed 4480/10000 samples...
Processed 5120/10000 samples...
Processed 5760/10000 samples...
Processed 6400/10000 samples...
Processed 7040/10000 samples...
Processed 7680/10000 samples...
Processed 8320/10000 samples...
Processed 8960/10000 samples...
Processed 9600/10000 samples...

Black-Box Evaluation (HopSkipJumpAttack)
Total samples evaluated:  10000
Samples eligible for attack (correct malicious): 2662

epsilon=0.05:
  Robustness:            56.39%
  Attack Success Rate:   43.61%
  Samples fooled:        1161 / 2662
  Confidence Drop (signed): -0.2253
  Confidence Drop (abs):    0.2259

epsilon=0.1:
  Robustness:            46.62%
  Attack Success Rate:   53.38%
  Samples fooled:        1421 / 2662
  Confidence Drop (signed): -0.2809
  Confidence Drop (abs):    0.2813

epsilon=0.2:
  Robu

ValueError: too many values to unpack (expected 2)

In [36]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
fgsm_hardened_model = DDoSDetector(num_classes=1).to(device)
fgsm_hardened_model.load_state_dict(torch.load("baseline_nids_cnn.pth", map_location=device))

results, adv_examples = black_box_evaluation(fgsm_hardened_model, X_test[:10000], y_test[:10000], epsilons=[0.05, 0.1, 0.2], batch_size=64)

Processed 640/10000 samples...
Processed 1280/10000 samples...
Processed 1920/10000 samples...
Processed 2560/10000 samples...
Processed 3200/10000 samples...
Processed 3840/10000 samples...
Processed 4480/10000 samples...
Processed 5120/10000 samples...
Processed 5760/10000 samples...
Processed 6400/10000 samples...
Processed 7040/10000 samples...
Processed 7680/10000 samples...
Processed 8320/10000 samples...
Processed 8960/10000 samples...
Processed 9600/10000 samples...

Black-Box Evaluation (HopSkipJumpAttack)
Total samples evaluated:  10000
Samples eligible for attack (correct malicious): 2693

epsilon=0.05:
  Robustness:            43.00%
  Attack Success Rate:   57.00%
  Samples fooled:        1535 / 2693
  Confidence Drop (signed): -0.2795
  Confidence Drop (abs):    0.2795

epsilon=0.1:
  Robustness:            41.44%
  Attack Success Rate:   58.56%
  Samples fooled:        1577 / 2693
  Confidence Drop (signed): -0.3008
  Confidence Drop (abs):    0.3008

epsilon=0.2:
  Robu

ValueError: too many values to unpack (expected 2)

## Blackbox Testing Models

In [ ]:
def pointwise_fgsm_attack(
    model,
    X_test,
    y_test,
    epsilon=0.1,
    num_features_to_perturb=None,  # How many features to perturb (None = all)
    feature_selection='gradient',   # 'gradient', 'random', or 'top_k'
    batch_size=512,
    device='cuda'
):
    """
    Pointwise FGSM Attack - Perturb only selected features
    
    Args:
        model: Trained model
        X_test: Test features (N, num_features)
        y_test: Test labels (N,)
        epsilon: Perturbation magnitude
        num_features_to_perturb: How many features to perturb per sample
                                 If None, perturb all features (standard FGSM)
        feature_selection: How to select features to perturb:
            - 'gradient': Features with highest gradient magnitude
            - 'random': Random features
            - 'top_k': Top-k most important features
        batch_size: Batch size for processing
        device: 'cuda' or 'cpu'
    
    Returns:
        adversarial_examples: Perturbed data
        feature_masks: Which features were perturbed (N, num_features)
        results: Dictionary with attack metrics
    """
    model = model.to(device)
    model.eval()
    
    num_samples = len(X_test)
    num_total_features = X_test.shape[1]
    
    # Default: perturb all features (standard FGSM)
    if num_features_to_perturb is None:
        num_features_to_perturb = num_total_features
    
    print(f"\n{'='*60}")
    print(f"Pointwise FGSM Attack")
    print(f"{'='*60}")
    print(f"Epsilon: {epsilon}")
    print(f"Feature selection: {feature_selection}")
    print(f"Features to perturb: {num_features_to_perturb}/{num_total_features}")
    print(f"Total samples: {num_samples}")
    print(f"{'='*60}\n")
    
    # Storage
    adversarial_examples = []
    feature_masks = []
    all_original_preds = []
    all_adv_preds = []
    
    criterion = nn.BCEWithLogitsLoss()
    num_batches = (num_samples + batch_size - 1) // batch_size
    
    for batch_idx in tqdm(range(num_batches), desc="Pointwise Attack"):
        start_idx = batch_idx * batch_size
        end_idx = min((batch_idx + 1) * batch_size, num_samples)
        
        X_batch = X_test[start_idx:end_idx].to(device)
        y_batch = y_test[start_idx:end_idx].to(device)
        
        # Enable gradients
        X_batch.requires_grad = True
        
        # Forward pass
        outputs = model(X_batch).squeeze(1)
        loss = criterion(outputs, y_batch.float())
        
        # Backward pass to get gradients
        model.zero_grad()
        loss.backward()
        
        # Get gradients w.r.t input
        data_grad = X_batch.grad.data  # (batch, num_features)
        
        # Original predictions
        with torch.no_grad():
            original_probs = torch.sigmoid(outputs)
            original_preds = (original_probs > 0.5).long()
        
        # ====================================================================
        # FEATURE SELECTION - Choose which features to perturb
        # ====================================================================
        
        batch_feature_masks = torch.zeros_like(X_batch, dtype=torch.bool)
        
        if feature_selection == 'gradient':
            # Perturb features with highest gradient magnitude
            grad_magnitude = data_grad.abs()  # (batch, num_features)
            
            for i in range(len(X_batch)):
                # Get top-k features by gradient magnitude
                _, top_indices = torch.topk(grad_magnitude[i], num_features_to_perturb)
                batch_feature_masks[i, top_indices] = True
        
        elif feature_selection == 'random':
            # Randomly select features to perturb
            for i in range(len(X_batch)):
                random_indices = torch.randperm(num_total_features)[:num_features_to_perturb]
                batch_feature_masks[i, random_indices] = True
        
        elif feature_selection == 'top_k':
            # Perturb top-k most important features (by absolute value)
            feature_importance = X_batch.abs()  # (batch, num_features)
            
            for i in range(len(X_batch)):
                _, top_indices = torch.topk(feature_importance[i], num_features_to_perturb)
                batch_feature_masks[i, top_indices] = True
        
        else:
            raise ValueError(f"Unknown feature_selection: {feature_selection}")
        
        # ====================================================================
        # CREATE ADVERSARIAL EXAMPLES - Only perturb selected features
        # ====================================================================
        
        # Standard FGSM perturbation
        perturbation = epsilon * data_grad.sign()
        
        # Apply mask - only perturb selected features
        perturbation = perturbation * batch_feature_masks.float()
        
        # Create adversarial examples
        X_adv = X_batch + perturbation
        X_adv = X_adv.detach()
        
        # Get adversarial predictions
        with torch.no_grad():
            adv_outputs = model(X_adv).squeeze(1)
            adv_probs = torch.sigmoid(adv_outputs)
            adv_preds = (adv_probs > 0.5).long()
        
        # Store results (move to CPU)
        adversarial_examples.append(X_adv.cpu())
        feature_masks.append(batch_feature_masks.cpu())
        all_original_preds.append(original_preds.cpu())
        all_adv_preds.append(adv_preds.cpu())
        
        # Clear GPU memory
        del X_batch, y_batch, outputs, loss, data_grad, perturbation, X_adv
        torch.cuda.empty_cache()
    
    # Concatenate results
    adversarial_examples = torch.cat(adversarial_examples, dim=0)
    feature_masks = torch.cat(feature_masks, dim=0)
    all_original_preds = torch.cat(all_original_preds, dim=0)
    all_adv_preds = torch.cat(all_adv_preds, dim=0)
    y_test_cpu = y_test.cpu()
    
    # ====================================================================
    # CALCULATE METRICS
    # ====================================================================
    
    original_accuracy = (all_original_preds == y_test_cpu).float().mean().item()
    adversarial_accuracy = (all_adv_preds == y_test_cpu).float().mean().item()
    
    correctly_classified = (all_original_preds == y_test_cpu)
    successful_attacks = (all_adv_preds != y_test_cpu) & correctly_classified
    asr = successful_attacks.float().sum().item() / correctly_classified.float().sum().item()
    
    # Perturbation statistics
    perturbations = (adversarial_examples - X_test.cpu()).abs()
    avg_perturbation = perturbations.mean().item()
    max_perturbation = perturbations.max().item()
    l2_perturbation = torch.norm(perturbations, p=2, dim=1).mean().item()
    
    # Feature-specific statistics
    num_features_perturbed_per_sample = feature_masks.float().sum(dim=1).mean().item()
    percent_features_perturbed = (num_features_perturbed_per_sample / num_total_features) * 100
    
    results = {
        'epsilon': epsilon,
        'feature_selection': feature_selection,
        'num_features_to_perturb': num_features_to_perturb,
        'original_accuracy': original_accuracy,
        'adversarial_accuracy': adversarial_accuracy,
        'asr': asr,
        'avg_perturbation': avg_perturbation,
        'max_perturbation': max_perturbation,
        'l2_perturbation': l2_perturbation,
        'avg_features_perturbed': num_features_perturbed_per_sample,
        'percent_features_perturbed': percent_features_perturbed,
        'num_samples': num_samples,
        'num_correctly_classified': correctly_classified.sum().item(),
        'num_successful_attacks': successful_attacks.sum().item()
    }
    
    # Print results
    print(f"\n{'='*60}")
    print("RESULTS")
    print(f"{'='*60}")
    print(f"Original Accuracy: {original_accuracy*100:.2f}%")
    print(f"Adversarial Accuracy: {adversarial_accuracy*100:.2f}%")
    print(f"Attack Success Rate (ASR): {asr*100:.2f}%")
    print(f"Avg Features Perturbed: {num_features_perturbed_per_sample:.1f} ({percent_features_perturbed:.1f}%)")
    print(f"Average L∞ Perturbation: {avg_perturbation:.6f}")
    print(f"Average L2 Perturbation: {l2_perturbation:.6f}")
    print(f"Successfully Attacked: {successful_attacks.sum().item()}/{correctly_classified.sum().item()}")
    print(f"{'='*60}\n")
    
    return adversarial_examples, feature_masks, results


def compare_pointwise_strategies(
    model,
    X_test,
    y_test,
    epsilon=0.1,
    feature_percentages=[10, 25, 50, 75, 100],
    device='cuda'
):
    """
    Compare different pointwise attack strategies
    
    Args:
        model: Trained model
        X_test, y_test: Test data
        epsilon: Perturbation magnitude
        feature_percentages: List of percentages of features to perturb
        device: 'cuda' or 'cpu'
    
    Returns:
        Comparison results
    """
    num_total_features = X_test.shape[1]
    strategies = ['gradient', 'random', 'top_k']
    
    all_results = {}
    
    print(f"\n{'='*70}")
    print(f"COMPARING POINTWISE ATTACK STRATEGIES")
    print(f"{'='*70}")
    print(f"Total features: {num_total_features}")
    print(f"Epsilon: {epsilon}")
    print(f"{'='*70}\n")
    
    for strategy in strategies:
        all_results[strategy] = {}
        
        for percent in feature_percentages:
            num_features = max(1, int(num_total_features * percent / 100))
            
            print(f"\n--- Strategy: {strategy.upper()}, Features: {percent}% ({num_features}) ---")
            
            _, _, results = pointwise_fgsm_attack(
                model=model,
                X_test=X_test,
                y_test=y_test,
                epsilon=epsilon,
                num_features_to_perturb=num_features,
                feature_selection=strategy,
                batch_size=512,
                device=device
            )
            
            all_results[strategy][percent] = results
    
    # Print comparison table
    print(f"\n{'='*90}")
    print("COMPARISON TABLE")
    print(f"{'='*90}")
    print(f"{'Strategy':<15} {'% Features':<15} {'ASR':<10} {'Adv Acc':<12} {'Avg Pert':<15}")
    print(f"{'-'*90}")
    
    for strategy in strategies:
        for percent in feature_percentages:
            res = all_results[strategy][percent]
            print(f"{strategy:<15} {percent:<15}% {res['asr']*100:<10.2f} "
                  f"{res['adversarial_accuracy']*100:<12.2f} {res['avg_perturbation']:<15.6f}")
    
    print(f"{'='*90}\n")
    
    return all_results


def analyze_important_features(
    model,
    X_test,
    y_test,
    epsilon=0.1,
    top_k=10,
    device='cuda'
):
    """
    Analyze which features are most important for successful attacks
    
    Returns:
        feature_importance: Array of size (num_features,) with importance scores
    """
    model = model.to(device)
    model.eval()
    
    num_features = X_test.shape[1]
    feature_attack_success = torch.zeros(num_features)
    feature_gradient_sum = torch.zeros(num_features)
    
    print(f"\n{'='*60}")
    print("ANALYZING FEATURE IMPORTANCE FOR ATTACKS")
    print(f"{'='*60}\n")
    
    criterion = nn.BCEWithLogitsLoss()
    
    # Process in batches
    batch_size = 512
    num_batches = (len(X_test) + batch_size - 1) // batch_size
    
    for batch_idx in tqdm(range(num_batches), desc="Analyzing"):
        start_idx = batch_idx * batch_size
        end_idx = min((batch_idx + 1) * batch_size, len(X_test))
        
        X_batch = X_test[start_idx:end_idx].to(device)
        y_batch = y_test[start_idx:end_idx].to(device)
        
        X_batch.requires_grad = True
        
        # Forward pass
        outputs = model(X_batch).squeeze(1)
        loss = criterion(outputs, y_batch.float())
        
        # Get gradients
        model.zero_grad()
        loss.backward()
        data_grad = X_batch.grad.data
        
        # Accumulate gradient magnitudes
        feature_gradient_sum += data_grad.abs().sum(dim=0).cpu()
        
        # Clear memory
        del X_batch, y_batch, outputs, loss, data_grad
        torch.cuda.empty_cache()
    
    # Average gradient magnitude per feature
    feature_importance = feature_gradient_sum / len(X_test)
    
    # Get top-k features
    top_k_values, top_k_indices = torch.topk(feature_importance, top_k)
    
    print(f"\nTop {top_k} Most Important Features for Attacks:")
    print(f"{'='*60}")
    print(f"{'Rank':<10} {'Feature Index':<20} {'Importance Score':<20}")
    print(f"{'-'*60}")
    
    for rank, (idx, value) in enumerate(zip(top_k_indices, top_k_values), 1):
        print(f"{rank:<10} {idx.item():<20} {value.item():<20.6f}")
    
    print(f"{'='*60}\n")
    
    return feature_importance.numpy()


# ============================================================================
# EXAMPLE USAGE
# ============================================================================

if __name__ == "__main__":
    import torch
    from torch.utils.data import DataLoader, TensorDataset
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    # Load your model and data
    # model = load_model('baseline_nids_cnn.pth')
    # X_test, y_test = load_test_data()
    
    # Example 1: Basic pointwise attack with gradient-based selection
    print("\n" + "="*70)
    print("EXAMPLE 1: Gradient-Based Feature Selection")
    print("="*70)
    
    X_adv, masks, results = pointwise_fgsm_attack(
        model=model,
        X_test=X_test,
        y_test=y_test,
        epsilon=0.1,
        num_features_to_perturb=10,  # Perturb only 10 features per sample
        feature_selection='gradient',
        batch_size=512,
        device=device
    )
    
    # Save adversarial examples
    torch.save({
        'X_adv': X_adv,
        'y_adv': y_test,
        'masks': masks,
        'epsilon': 0.1,
        'num_features_perturbed': 10,
        'selection_method': 'gradient',
        'results': results
    }, 'pointwise_adversarial_examples.pt')
    
    
    # Example 2: Compare different strategies
    print("\n" + "="*70)
    print("EXAMPLE 2: Compare Feature Selection Strategies")
    print("="*70)
    
    comparison_results = compare_pointwise_strategies(
        model=model,
        X_test=X_test,
        y_test=y_test,
        epsilon=0.1,
        feature_percentages=[10, 25, 50, 75, 100],
        device=device
    )
    
    
    # Example 3: Analyze feature importance
    print("\n" + "="*70)
    print("EXAMPLE 3: Feature Importance Analysis")
    print("="*70)
    
    feature_importance = analyze_important_features(
        model=model,
        X_test=X_test,
        y_test=y_test,
        epsilon=0.1,
        top_k=20,
        device=device
    )
    
    # Save feature importance
    np.save('feature_importance.npy', feature_importance)

In [83]:
def boundary_attack_blackbox(
    model,
    X_test,
    y_test,
    num_samples=1000,
    steps=25000,
    batch_size=32,
    device='cuda'
):
    model = model.to(device)
    model.eval()
    
    # Wrap model
    wrapped_model = DDoSDetectorBinaryWrapper(model).to(device)
    wrapped_model.eval()
    
    # Create foolbox model
    fmodel = fb.PyTorchModel(wrapped_model, bounds=(X_test.min().item(), X_test.max().item()))
    
    # Initialize Boundary Attack
    attack = BoundaryAttack(steps=steps)
    
    print(f"\n{'='*70}")
    print(f"BLACK-BOX BOUNDARY ATTACK")
    print(f"{'='*70}")
    print(f"Attack parameters:")
    print(f"  - Samples: {num_samples}")
    print(f"  - Steps: {steps}")
    print(f"  - Type: Black-box (query-based)")
    print(f"{'='*70}\n")
    
    # Limit to num_samples
    if num_samples < len(X_test):
        indices = np.random.choice(len(X_test), num_samples, replace=False)
        X_subset = X_test[indices]
        y_subset = y_test[indices]
    else:
        X_subset = X_test
        y_subset = y_test
    
    all_original_preds = []
    all_adv_preds = []
    all_distances = []
    all_success = []
    all_queries = []
    
    # Process in smaller batches
    num_batches = (len(X_subset) + batch_size - 1) // batch_size
    
    for batch_idx in tqdm(range(num_batches), desc="Attacking batches"):
        start_idx = batch_idx * batch_size
        end_idx = min((batch_idx + 1) * batch_size, len(X_subset))
        
        X_batch = X_subset[start_idx:end_idx].to(device)
        y_batch = y_subset[start_idx:end_idx].to(device)
        
        # Get original predictions
        with torch.no_grad():
            original_outputs = model(X_batch)
            original_probs = torch.sigmoid(original_outputs).squeeze()
            original_preds = (original_probs > 0.5).long()
        
        # Only attack correctly classified samples
        correct_mask = (original_preds == y_batch)
        
        if correct_mask.sum() == 0:
            continue
        
        X_correct = X_batch[correct_mask]
        y_correct = y_batch[correct_mask]
        original_preds_correct = original_preds[correct_mask]
        
        # Perform Boundary Attack
        _, adversarials, success = attack(
            fmodel,
            X_correct,
            y_correct, 
            epsilons=None
        )
        
        # Get adversarial predictions
        with torch.no_grad():
            adv_outputs = model(adversarials)
            adv_probs = torch.sigmoid(adv_outputs).squeeze()
            adv_preds = (adv_probs > 0.5).long()
        
        # Calculate L2 distance
        distances = torch.norm(adversarials - X_correct, p=2, dim=1)
        
        # Store results
        all_original_preds.extend(original_preds_correct.cpu().numpy())
        all_adv_preds.extend(adv_preds.cpu().numpy())
        all_distances.extend(distances.cpu().numpy())
        all_success.extend(success.cpu().numpy())
        
        # Clear GPU cache
        del X_batch, y_batch, adversarials
        torch.cuda.empty_cache()
    
    # Calculate metrics
    all_original_preds = np.array(all_original_preds)
    all_adv_preds = np.array(all_adv_preds)
    all_distances = np.array(all_distances)
    all_success = np.array(all_success)
    
    # Attack Success Rate
    asr = all_success.mean()
    
    # Average L2 distance for successful attacks
    successful_distances = all_distances[all_success]
    avg_distance = successful_distances.mean() if len(successful_distances) > 0 else 0
    
    results = {
        'asr': asr,
        'avg_l2_distance': avg_distance,
        'num_samples': len(all_original_preds),
        'num_successful': all_success.sum(),
        'distances': all_distances,
        'success': all_success
    }
    
    # Print results
    print(f"\n{'='*70}")
    print(f"BOUNDARY ATTACK RESULTS")
    print(f"{'='*70}")
    print(f"Samples attacked: {len(all_original_preds)}")
    print(f"Successful attacks: {all_success.sum()} / {len(all_original_preds)}")
    print(f"Attack Success Rate (ASR): {asr*100:.2f}%")
    print(f"Average L2 Distance: {avg_distance:.4f}")
    if len(successful_distances) > 0:
        print(f"Min L2 Distance: {successful_distances.min():.4f}")
        print(f"Max L2 Distance: {successful_distances.max():.4f}")
        print(f"Median L2 Distance: {np.median(successful_distances):.4f}")
    print(f"{'='*70}\n")
    
    return results

In [88]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# # Load test data
# clean_data = torch.load('clean_data.pt')
# X_test = clean_data['X_test']
# y_test = clean_data['y_test']

# Load baseline model
baseline_model = load_model('baseline_nids_cnn.pth', device=device)

✓ Loaded baseline model from baseline_nids_cnn.pth


In [89]:
# Attack (start SMALL)
results = boundary_attack_blackbox(
    model=baseline_model,
    X_test=X_test,
    y_test=y_test,
    num_samples=100,   # ← Start with 100 samples
    steps=10000,       # ← Fewer steps for testing
    batch_size=16,
    device=device
)

print(f"ASR: {results['asr']*100:.2f}%")


BLACK-BOX BOUNDARY ATTACK
Attack parameters:
  - Samples: 100
  - Steps: 10000
  - Type: Black-box (query-based)



Attacking batches:   0%|          | 0/7 [00:00<?, ?it/s]

Attacking batches:  57%|█████▋    | 4/7 [01:31<01:08, 23.00s/it]


ValueError: init_attack failed for 1.0 of 16 inputs

In [ ]:
def boundary_attack_blackbox(model, X_test, y_test, num_samples=100, 
                             steps=5000, batch_size=16, device='cuda'):
    """
    Boundary Attack with proper adversarial initialization
    """
    from foolbox.attacks import BoundaryAttack
    from foolbox import PyTorchModel
    
    model.eval()
    
    # Move y_test to device for comparison
    y_test_device = y_test.to(device)
    
    # Create Foolbox model
    fmodel = PyTorchModel(model, bounds=(0, 1))
    
    # Select subset
    indices = torch.randperm(len(X_test))[:num_samples]
    X_batch = X_test[indices].to(device)
    y_batch = y_test[indices].to(device)
    
    # Get correct predictions only
    with torch.no_grad():
        original_preds = model(X_batch).argmax(dim=1)
    correct_mask = (original_preds == y_batch)
    X_correct = X_batch[correct_mask]
    y_correct = y_batch[correct_mask]
    
    print(f"Using {len(X_correct)} correctly classified samples")
    
    # Generate adversarial starting points 
    starting_points = []
    
    for y_true in y_correct:
        # Find samples from other classes (compare on same device)
        other_class_indices = torch.where(y_test_device != y_true)[0]
        
        if len(other_class_indices) > 0:
            # Pick a random sample from another class
            idx = other_class_indices[torch.randint(len(other_class_indices), (1,))].item()
            starting_points.append(X_test[idx].to(device))
        else:
            # Fallback: random noise
            starting_points.append(torch.rand_like(X_correct[0]))
    
    starting_points = torch.stack(starting_points)
    
    # Verify starting points are adversarial
    with torch.no_grad():
        start_preds = model(starting_points).argmax(dim=1)
        adv_mask = start_preds != y_correct
        print(f"Adversarial starting points: {adv_mask.sum()}/{len(adv_mask)}")
    
    # Create the attack
    attack = BoundaryAttack(steps=steps)
    
    # Run attack
    _, adversarials, success = attack(
        fmodel,
        X_correct,
        y_correct,
        epsilons=0.3,
        starting_points=starting_points
    )
    
    # Calculate metrics
    asr = success.float().mean().item()
    
    print(f"✓ Attack completed")
    print(f"  ASR: {asr*100:.2f}%")
    print(f"  Successful: {success.sum().item()}/{len(success)}")
    
    return {
        'adversarials': adversarials,
        'success': success,
        'asr': asr,
        'original': X_correct,
        'labels': y_correct
    }

In [109]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# Attack (start SMALL)
results = boundary_attack_blackbox(
    model=baseline_model,
    X_test=X_test,
    y_test=y_test,
    num_samples=100,   # ← Start with 100 samples
    steps=10000,       # ← Fewer steps for testing
    batch_size=16,
    device=device
)


Using 26 correctly classified samples
Adversarial starting points: 0/26


ValueError: 26.0 of 26 starting_points are not adversarial

In [ ]:

def boundary_attack_comparison(
    baseline_model_path,
    finetuned_model_path,
    X_test,
    y_test,
    num_samples=500,
    steps=25000,
    device='cuda'
):
    """
    Compare baseline vs finetuned model against Boundary Attack
    
    Args:
        baseline_model_path: Path to baseline model .pth
        finetuned_model_path: Path to finetuned model .pth
        X_test: Test features
        y_test: Test labels
        num_samples: Number of samples to attack
        steps: Attack steps
        device: 'cuda' or 'cpu'
    
    Returns:
        Comparison results
    """
    print(f"\n{'='*70}")
    print("BASELINE MODEL - BLACK-BOX ATTACK")
    print(f"{'='*70}")
    
    baseline_model = load_model(baseline_model_path, num_classes=1, device=device)
    baseline_results = boundary_attack_blackbox(
        baseline_model,
        X_test,
        y_test,
        num_samples=num_samples,
        steps=steps,
        device=device
    )
    
    print(f"\n{'='*70}")
    print("FINETUNED MODEL - BLACK-BOX ATTACK")
    print(f"{'='*70}")
    
    finetuned_model = load_model(finetuned_model_path, num_classes=1, device=device)
    finetuned_results = boundary_attack_blackbox(
        finetuned_model,
        X_test,
        y_test,
        num_samples=num_samples,
        steps=steps,
        device=device
    )
    
    # Comparison
    print(f"\n{'='*70}")
    print("COMPARISON: BASELINE vs FINETUNED")
    print(f"{'='*70}")
    print(f"{'Metric':<30} {'Baseline':<15} {'Finetuned':<15} {'Improvement':<15}")
    print(f"{'-'*70}")
    
    baseline_asr = baseline_results['asr'] * 100
    finetuned_asr = finetuned_results['asr'] * 100
    asr_improvement = baseline_asr - finetuned_asr
    
    print(f"{'ASR (%)':<30} {baseline_asr:<15.2f} {finetuned_asr:<15.2f} {asr_improvement:<15.2f}")
    
    baseline_dist = baseline_results['avg_l2_distance']
    finetuned_dist = finetuned_results['avg_l2_distance']
    dist_change = finetuned_dist - baseline_dist
    
    print(f"{'Avg L2 Distance':<30} {baseline_dist:<15.4f} {finetuned_dist:<15.4f} {dist_change:<15.4f}")
    print(f"{'-'*70}")
    print(f"\nInterpretation:")
    if asr_improvement > 0:
        print(f"✓ Finetuned model is MORE ROBUST (ASR reduced by {asr_improvement:.2f}%)")
    else:
        print(f"✗ Finetuned model is LESS ROBUST (ASR increased by {abs(asr_improvement):.2f}%)")
    
    if dist_change > 0:
        print(f"✓ Finetuned model requires LARGER perturbations (+{dist_change:.4f})")
    else:
        print(f"✗ Finetuned model requires SMALLER perturbations ({dist_change:.4f})")
    print(f"{'='*70}\n")
    
    return {
        'baseline': baseline_results,
        'finetuned': finetuned_results
    }


# ============================================================================
# EXAMPLE USAGE
# ============================================================================

if __name__ == "__main__":
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    # Load test data
    clean_data = torch.load('clean_data.pt')
    X_test = clean_data['X_test']
    y_test = clean_data['y_test']
    
    print(f"Test set size: {len(X_test)}")
    
    # Option 1: Attack only the finetuned model
    print("\n" + "="*70)
    print("OPTION 1: Attack Finetuned Model Only")
    print("="*70)
    
    finetuned_model = load_model('finetuned_nids_cnn.pth', num_classes=1, device=device)
    
    results = boundary_attack_blackbox(
        model=finetuned_model,
        X_test=X_test,
        y_test=y_test,
        num_samples=1000,  # Start small, Boundary is SLOW!
        steps=25000,       # Default steps
        batch_size=32,
        device=device
    )
    
    # Save results
    torch.save(results, 'boundary_attack_results.pt')
    print("✓ Results saved to boundary_attack_results.pt")
    
    
    # Option 2: Compare baseline vs finetuned
    print("\n" + "="*70)
    print("OPTION 2: Compare Baseline vs Finetuned")
    print("="*70)
    
    comparison = boundary_attack_comparison(
        baseline_model_path='baseline_nids_cnn.pth',
        finetuned_model_path='finetuned_nids_cnn.pth',
        X_test=X_test,
        y_test=y_test,
        num_samples=500,   # Even smaller for comparison
        steps=25000,
        device=device
    )
    
    # Save comparison
    torch.save(comparison, 'boundary_attack_comparison.pt')
    print("✓ Comparison saved to boundary_attack_comparison.pt")

In [ ]:
def compute_adaptive_weights(student_logits_2d, teacher_logits_2d_list):
    """
    Computes normalised per-teacher importance weights for each sample.
 
    Args:
        student_logits_2d:       (B, 2)  student wrapped logits
        teacher_logits_2d_list:  list of (B, 2) teacher wrapped logits
 
    Returns:
        normalized_weights:  (num_teachers, B)
        Each column sums to 1.0 — tells us how much to trust each teacher
        for that particular traffic sample.
 
    Step-by-step:
        1. cosine_similarity(student, teacher_i) → scalar per sample ∈ [−1, 1]
        2. weight_i = 1 + cosine_sim  →  shifts range to [0, 2] (no negative weights)
        3. Normalize across teachers so weights sum to 1.0 per sample
    """
    # Step 1: cosine similarity between student and each teacher — shape (num_teachers, B)
    similarities = torch.stack([
        F.cosine_similarity(student_logits_2d, t_logits, dim=1)   # (B,)
        for t_logits in teacher_logits_2d_list
    ])  # → (num_teachers, B)
 
    # Step 2: shift to positive range [0, 2]
    weights = 1.0 + similarities   # (num_teachers, B)
 
    # Step 3: normalise so columns sum to 1
    weights_sum = weights.sum(dim=0, keepdim=True)   # (1, B)
    normalized = weights / (weights_sum + 1e-8)       # (num_teachers, B)
 
    return normalized
 

In [ ]:
def compute_weighted_soft_targets(teacher_logits_2d_list, normalized_weights, temperature):
    """
    Produces p' — the teacher consensus soft target for each sample.
 
    For each teacher:
      1. Softmax at temperature τ → soft probability distribution over {Attack, Benign}
      2. Multiply each sample's distribution by that teacher's weight for that sample
      3. Sum across teachers → weighted consensus distribution p'
 
    Args:
        teacher_logits_2d_list:  list of (B, 2) tensors
        normalized_weights:      (num_teachers, B)
        temperature:             τ scalar
 
    Returns:
        p_prime:  (B, 2)  — weighted soft targets ready for KL divergence
    """
    device = teacher_logits_2d_list[0].device
    B = teacher_logits_2d_list[0].size(0)
    p_prime = torch.zeros(B, 2, device=device)
 
    for i, t_logits in enumerate(teacher_logits_2d_list):
        soft = F.softmax(t_logits / temperature, dim=1)   # (B, 2)
        w = normalized_weights[i].unsqueeze(1)             # (B, 1)
        p_prime += w * soft
 
    return p_prime   # (B, 2)

In [ ]:
def distillation_loss_kld(student_logits_2d, p_prime, temperature):
    """
    KL Divergence between student soft outputs and weighted teacher soft targets.
    Scaled by τ² to compensate for the gradient magnitude reduction at high temperature.
 
    KLD(p' || q)   where:
      p' = weighted teacher soft targets  (already softmaxed in compute_weighted_soft_targets)
      q  = student soft output at temperature τ  (log-softmax for KLDivLoss)
    """
    q_log = F.log_softmax(student_logits_2d / temperature, dim=1)  # (B, 2)
    kld = F.kl_div(q_log, p_prime, reduction="batchmean")
    return kld * (temperature ** 2)
 
 
def student_bce_loss(student_raw_logit, labels_float):
    """
    Your original BCE loss — keeps the student grounded in ground truth.
    student_raw_logit: (B, 1)  raw output from DDoSDetector (not wrapped)
    labels_float:      (B,)    float labels
    """
    return F.binary_cross_entropy_with_logits(
        student_raw_logit.squeeze(1), labels_float
    )

In [ ]:
def distillation_loss_kld(student_logits_2d, p_prime, temperature):
    """
    KL Divergence between student soft outputs and weighted teacher soft targets.
    Scaled by τ² to compensate for the gradient magnitude reduction at high temperature.
 
    KLD(p' || q)   where:
      p' = weighted teacher soft targets  (already softmaxed in compute_weighted_soft_targets)
      q  = student soft output at temperature τ  (log-softmax for KLDivLoss)
    """
    q_log = F.log_softmax(student_logits_2d / temperature, dim=1)  # (B, 2)
    kld = F.kl_div(q_log, p_prime, reduction="batchmean")
    return kld * (temperature ** 2)
 
 
def student_bce_loss(student_raw_logit, labels_float):
    """
    Your original BCE loss — keeps the student grounded in ground truth.
    student_raw_logit: (B, 1)  raw output from DDoSDetector (not wrapped)
    labels_float:      (B,)    float labels
    """
    return F.binary_cross_entropy_with_logits(
        student_raw_logit.squeeze(1), labels_float
    )

In [ ]:
def load_teacher(path, device):
    """
    Loads a pre-trained DDoSDetector teacher and freezes all its parameters.
    The teacher is wrapped in DDoSDetectorBinaryWrapper for distillation use.
    """
    model = DDoSDetector(num_classes=1).to(device)
    model.load_state_dict(torch.load(path, map_location=device))
    model.eval()
    for param in model.parameters():
        param.requires_grad = False  # teachers NEVER update during distillation
    wrapped = DDoSDetectorBinaryWrapper(model).to(device)
    wrapped.eval()
    print(f"  ✓ Loaded & frozen: {path}")
    return wrapped   # returns wrapped version for distillation
 
 
def load_student(device):
    """
    Initialises a fresh DDoSDetector student with the same architecture.
    Student starts from random weights — it will learn from teachers + clean data.
    """
    model = DDoSDetector(num_classes=1).to(device)
    print(f"  ✓ Student initialised (fresh weights, same architecture as teachers)")
    return model

In [ ]:
def train_one_epoch(student, student_wrapper,
                    teachers_wrapped,
                    train_loader, optimizer, cfg, device):
    """
    One epoch of adaptive multi-teacher knowledge distillation.
 
    Key design choices:
      - student trains on CLEAN data only (no adversarial samples)
      - teachers are frozen (eval mode, no gradients tracked)
      - cosine similarity weights are recomputed per batch — they change
        as the student learns, so the "best teacher" for a given sample
        can shift over training
      - student_logits.detach() is used when computing cosine weights so
        that the weight computation doesn't create unwanted gradient paths
    """
    student.train()
    student_wrapper.train()
    for t in teachers_wrapped:
        t.eval()   # ensure teachers stay frozen even if called in train context
 
    total_loss_sum = 0.0
    bce_sum = 0.0
    kld_sum = 0.0
    correct = 0
    total = 0
 
    pbar = tqdm(train_loader, desc="  Distilling", leave=False)
 
    for X_batch, y_batch in pbar:
        X_batch  = X_batch.to(device)
        y_batch  = y_batch.to(device)           # float labels (your format)
 
        optimizer.zero_grad()
 
        # ── Student forward pass ──────────────────────────────────────────
        student_raw_logit    = student(X_batch)          # (B, 1) — for BCE
        student_logits_2d    = student_wrapper(X_batch)  # (B, 2) — for KLD + cosine
 
        # ── Teacher forward passes (no gradient tracking) ─────────────────
        with torch.no_grad():
            teacher_logits_2d_list = [t(X_batch) for t in teachers_wrapped]
 
        # ── Adaptive weights via cosine similarity ─────────────────────────
        # We detach student_logits_2d here so the weight computation is
        # treated as a constant — it reflects current alignment but its
        # gradient path doesn't interfere with the main loss backprop.
        normalized_weights = compute_adaptive_weights(
            student_logits_2d.detach(),
            teacher_logits_2d_list
        )  # (num_teachers, B)
 
        # ── Weighted teacher consensus soft targets p' ─────────────────────
        p_prime = compute_weighted_soft_targets(
            teacher_logits_2d_list,
            normalized_weights,
            cfg["temperature"]
        )  # (B, 2)
 
        # ── Total loss ────────────────────────────────────────────────────
        loss, bce_val, kld_val = total_distillation_loss(
            student_raw_logit,
            student_logits_2d,
            p_prime,
            y_batch,            # your labels are already float
            cfg["temperature"],
            cfg["alpha"]
        )
 
        loss.backward()
        optimizer.step()
 
        # ── Track metrics ─────────────────────────────────────────────────
        total_loss_sum += loss.item()
        bce_sum  += bce_val
        kld_sum  += kld_val
 
        with torch.no_grad():
            probs = torch.sigmoid(student_raw_logit.squeeze(1))
            preds = (probs > 0.5).long()
            correct += (preds == y_batch.long()).sum().item()
            total   += len(y_batch)
 
        pbar.set_postfix({
            "loss": f"{loss.item():.4f}",
            "bce":  f"{bce_val:.4f}",
            "kld":  f"{kld_val:.4f}",
        })
 
    n = len(train_loader)
    return (total_loss_sum / n,
            bce_sum / n,
            kld_sum / n,
            correct / total)

In [ ]:
def evaluate_model(model, loader, device, label=""):
    """
    Standard evaluation on a DataLoader.
    Returns accuracy, precision, recall, F1 (binary).
    Matches your existing evaluate_model() function format.
    """
    model.eval()
    all_preds, all_labels = [], []
 
    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch = X_batch.to(device)
            outputs = model(X_batch).squeeze(1)
            probs   = torch.sigmoid(outputs)
            preds   = (probs > 0.5).long()
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y_batch.long().numpy())
 
    acc  = accuracy_score(all_labels, all_preds) * 100
    prec = precision_score(all_labels, all_preds, zero_division=0)
    rec  = recall_score(all_labels, all_preds, zero_division=0)
    f1   = f1_score(all_labels, all_preds, zero_division=0)
 
    if label:
        print(f"\n{'='*55}")
        print(f"  {label}")
        print(f"{'='*55}")
        print(f"  Accuracy:  {acc:.2f}%")
        print(f"  Precision: {prec:.4f}")
        print(f"  Recall:    {rec:.4f}")
        print(f"  F1 Score:  {f1:.4f}")
        print(f"{'='*55}")
 
    return {"accuracy": acc, "precision": prec, "recall": rec, "f1": f1}

In [ ]:
def evaluate_under_attack(model, loader, attack_fn, attack_kwargs,
                           device, label=""):
    """
    Evaluates the student on adversarially perturbed samples.
    Uses your attack style (gradient-based, applied per batch).
    Student was NEVER trained on these — this is the key test.
    """
    model.eval()
    all_preds, all_labels = [], []
    criterion = nn.BCEWithLogitsLoss()
 
    for X_batch, y_batch in loader:
        X_batch  = X_batch.to(device)
        y_batch  = y_batch.to(device)
 
        # Generate adversarial batch
        X_adv = attack_fn(model, X_batch, y_batch.float(), criterion,
                          **attack_kwargs)
 
        with torch.no_grad():
            outputs = model(X_adv).squeeze(1)
            probs   = torch.sigmoid(outputs)
            preds   = (probs > 0.5).long()
 
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(y_batch.long().cpu().numpy())
 
    acc  = accuracy_score(all_labels, all_preds) * 100
    prec = precision_score(all_labels, all_preds, zero_division=0)
    rec  = recall_score(all_labels, all_preds, zero_division=0)
    f1   = f1_score(all_labels, all_preds, zero_division=0)
 
    if label:
        print(f"\n{'='*55}")
        print(f"  {label}")
        print(f"{'='*55}")
        print(f"  Accuracy:  {acc:.2f}%")
        print(f"  Precision: {prec:.4f}")
        print(f"  Recall:    {rec:.4f}")
        print(f"  F1 Score:  {f1:.4f}")
        print(f"{'='*55}")
 
    return {"accuracy": acc, "precision": prec, "recall": rec, "f1": f1}